<div style="padding:28px;background:#142e42;border-radius:14px"><p style="color:#91ded1;letter-spacing:2px">SQUARES IN SQUARES</p><h1 style="color:white">CUDA Packing Search</h1><p style="color:#e4f2f5;font-size:19px">Explore contacts in parallel. Restore feasibility. Compose useful blocks.</p></div>

**Certified score: 22.303399758060. Lower is better.** Previous scored V23: **22.303884530818**.

Kirill's new Projected Union suggests a useful admission change: a near-feasible proposal may still beat its incumbent after LP restoration. This release combines credited public witnesses, tests that idea with our CUDA C++ optimizer, and recomposes the resulting block library. The ledger measures these stages separately.

Default **Run All** restores and certifies the frozen result on CPU. Optional GPU search is disabled. All search source is present as readable notebook cells.

In [ ]:
from pathlib import Path
import json,hashlib,shutil,sys,subprocess,math,time
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display,FileLink
RUN_SEARCH=False
roots=list(Path('/kaggle/input').rglob('projected_manifest.json'))if Path('/kaggle/input').exists()else []
DATA=roots[0].parent if roots else Path('inputs')
manifest=json.loads((DATA/'projected_manifest.json').read_text())
for name,digest in manifest.items():assert hashlib.sha256((DATA/name).read_bytes()).hexdigest()==digest,name
summary=json.loads((DATA/'projected_summary.json').read_text());trials=json.loads((DATA/'projected_trials.json').read_text());budget=json.loads((DATA/'projected_budget.json').read_text())
shutil.copyfile(DATA/'projected_submission.csv','submission.csv')
assert hashlib.sha256(Path('submission.csv').read_bytes()).hexdigest()==summary['sha256']
plt.rcParams.update({'figure.dpi':130,'font.family':'DejaVu Sans','axes.spines.top':False,'axes.spines.right':False,'axes.titleweight':'bold'})
print(f"Frozen score: {summary['score']:.12f}. {summary['puzzles']} explicit witnesses; {1000-summary['puzzles']} official grid fallbacks.")
print('CSV SHA256:',summary['sha256'])

## 1. Keep adoption and further search separate

[Kirill's V20](https://www.kaggle.com/code/kirilltym/squares-in-squares-cpu-cuda-mps-eda-projunion?scriptVersionId=361930202) retains our V23 compositions and contributes complementary CPU witnesses. [prvsiyan's V4](https://www.kaggle.com/code/prvsiyan/auto-updater?scriptVersionId=361948698) supplies additional whole-puzzle witnesses with per-size credits. The union selects complete layouts; it never averages coordinates. Their discoveries retain their attribution.

The following two stages measure further CUDA search and recomposition against that union. A candidate accepted during search may duplicate an adopted public witness; only the final admission ledger counts its additional score reduction.

In [ ]:
ledger=pd.DataFrame(summary['stages']);display(ledger.style.format({'score':'{:.12f}','gain':'{:.12f}'}))
fig,axes=plt.subplots(1,2,figsize=(13,4.4));labels=['Kirill witnesses','Other public witnesses','CUDA proposals + LP','CUDA recomposition'];vals=ledger.gain.iloc[1:].to_numpy()
axes[0].barh(labels,vals,color=['#9aafbd','#b5c5cc','#087f8c','#d78c50']);axes[0].invert_yaxis();axes[0].set(xlabel='Certified score reduction',title='Sequential contribution ledger')
for i,g in enumerate(vals):axes[0].text(g+max(vals)*.025,i,f'{g:.8f}',va='center',fontsize=9)
axes[0].set_xlim(0,max(vals)*1.35)
changes=pd.DataFrame(summary['own_changes'])
if len(changes):
    changes=changes.groupby('n',as_index=False).gain.sum().sort_values('gain',ascending=False);top=changes.head(12);axes[1].bar(top.n.astype(str),top.gain,color='#087f8c')
axes[1].set(xlabel='Puzzle n',ylabel='Score reduction after public union',title='Additional accepted gains');axes[1].ticklabel_format(axis='y',style='sci',scilimits=(0,0));fig.tight_layout();fig.savefig('projected_gains.png');plt.show()

## 2. Let LP repair a promising contact pattern

For square angles fixed by a CUDA proposal, choose one separating face normal $u_{ij}$ per nearby pair. Translation and container excess $f$ then enter linear constraints:

$$u_{ij}\cdot(c_j-c_i)\ge h_i(u_{ij})+h_j(u_{ij})+\varepsilon,\qquad
h_i\le c_{i,x}\le w+f-h_i,\quad h_i\le c_{i,y}\le h+f-h_i.$$

Minimize $f$ inside bounded centre movements. The LP may enlarge a compressed proposal while still beating its original incumbent. Unlike the previous early rejection gate, an overlapping proposal reaches this restoration attempt. The selected branch remains local: this is a search over contact patterns, not a convex formulation of the entire packing problem.

The new pilot uses **32 FP64 trajectories per puzzle**, eight compression depths, coherent patch rotations, opposite displacements across a fault and independent perturbations. The existing CUDA C++ L-BFGS kernel performs the relaxation. Only four selected candidates per puzzle enter CPU LP restoration, then all-pair repair and interval certification. No claim of a new global optimum or a fundamentally new packing family follows from these local moves.

In [ ]:
history=pd.DataFrame(trials['records']);display(history[['n','before','after','gain','direct_wins','lp_wins']].sort_values('gain',ascending=False).head(18).style.format({'before':'{:.9f}','after':'{:.9f}','gain':'{:.9f}'}))
display(pd.DataFrame(budget['stages']));print('Campaign wall time:',round(budget['seconds'],1),'seconds. This includes CPU restoration and certification.')
print('Gradient and square-endpoint checks:',trials['checks'])
print('Measured CUDA-kernel time for proposal relaxation:',round(trials['cuda_seconds'],2),'seconds; it is not the entire campaign time.')
history.to_csv('projected_trial_audit.csv',index=False)

## 3. Inspect the accepted geometry

Equal axes and unchanged unit sides make these panels comparable. Blue squares are tilted; grey squares are close to upright. The baseline is the refreshed public union, so only further accepted gains appear here.

In [ ]:
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon,Rectangle
def load(path):
    df=pd.read_csv(path,float_precision='round_trip');return {int(n):g.sort_values('square_id')[['x','y','angle']].to_numpy(float)for n,g in df.groupby('n')}
def side(p):
    e=.5*(abs(np.cos(p[:,2]))+abs(np.sin(p[:,2])));return float(np.max((p[:,:2]+e[:,None]).max(0)-(p[:,:2]-e[:,None]).min(0)))
def draw(ax,p,L,title):
    u=np.array([[-.5,-.5],[.5,-.5],[.5,.5],[-.5,.5]]);c=np.cos(p[:,2]);s=np.sin(p[:,2]);v=p[:,None,:2]+np.stack([c[:,None]*u[:,0]-s[:,None]*u[:,1],s[:,None]*u[:,0]+c[:,None]*u[:,1]],2);v-=v.reshape(-1,2).min(0)
    angle=abs((p[:,2]+np.pi/4)%(np.pi/2)-np.pi/4);ax.add_collection(PatchCollection([Polygon(q)for q in v],facecolors=np.where(angle[:,None]>.001,[.08,.48,.58,.86],[.75,.79,.82,.86]),edgecolors='#27404c',linewidths=.2));ax.add_patch(Rectangle((0,0),side(p),side(p),fill=False,ec='#c65d45',lw=1.3));ax.set(xlim=(-.2,L+.2),ylim=(-.2,L+.2),aspect='equal',title=title);ax.set_xticks([]);ax.set_yticks([])
before=load(DATA/'projected_public.csv');after=load('submission.csv');chosen=changes.n.head(2).tolist()
fig,axes=plt.subplots(len(chosen),2,figsize=(11,5*len(chosen)),squeeze=False)
for row,n in enumerate(chosen):
    L=side(before[n]);draw(axes[row,0],before[n],L,f'n={n} · Public union · L={L:.8f}');draw(axes[row,1],after[n],L,f'n={n} · CUDA + restoration · L={side(after[n]):.8f}')
fig.tight_layout();fig.savefig('projected_layouts.png');plt.show()

## 4. Readable implementation

The following cells write the complete source files without running a search. CUDA C++ kernels are included inside their Python modules. These retain the earlier support-profile, staircase DP and geometry implementations; the new projected-search runner follows them. Default replay requires NumPy, pandas, SciPy, matplotlib and compatible mpmath. Optional search additionally requires CuPy, PyTorch and an NVIDIA GPU.

The DP minimizes the maximum block excess along a staircase. Conservative support profiles permit distinct separating normals at successive joins. Whole-puzzle proposals are re-cropped into reusable blocks: this is why a local change may help another size. The composer is the previously published method; refreshed inputs and its measured gains are recorded separately.

In [ ]:
for name in ['research','artifacts','research/top1-refresh/kirill_sources','research/refresh-downloads/update25-kirilltym/experiment_logs']:Path(name).mkdir(exist_ok=True,parents=True)
for name in ['SOURCE_NOTICES.md','projected_SOURCE_NOTICES.md','projected_kirill_NOTICES.md','projected_prvsiyan_credits.csv']:shutil.copyfile(DATA/name,name)
shutil.copyfile(DATA/'projected_target_seed.json','research/refresh-downloads/update25-kirilltym/experiment_logs/round22_summary.json')

### solver.py

In [ ]:
%%writefile solver.py
"""Constructive square packing with rigid-block linear compaction."""
import math
import numpy as np
from scipy.optimize import linprog

GAP = 2e-7

def side(p):
    h=(np.abs(np.cos(p[:,2]))+np.abs(np.sin(p[:,2])))/2
    lo=(p[:,:2]-h[:,None]).min(axis=0)
    hi=(p[:,:2]+h[:,None]).max(axis=0)
    return float(np.max(hi-lo))

def normalize(p, margin=GAP):
    p=np.asarray(p,dtype=float).copy()
    p[:,:2]*=1+margin
    h=(np.abs(np.cos(p[:,2]))+np.abs(np.sin(p[:,2])))/2
    p[:,:2]-=(p[:,:2]-h[:,None]).min(axis=0)
    return p

def grid(n):
    k=math.ceil(math.sqrt(n))
    return np.array([(i%k+.5,i//k+.5,0.) for i in range(n)])

def strip_blocks(k,w,r,m,theta=math.pi/4):
    L=k+w/math.sqrt(2)
    tri=np.array([(i+.5,j+.5) for i in range(k-1) for j in range(k-1-i)])
    small=np.array([(i+.5,j+.5) for i in range(r) for j in range(r-i)]).reshape(-1,2)
    blocks=[tri,L-tri]
    if len(small):
        blocks += [np.column_stack((L-small[:,0],small[:,1])),np.column_stack((small[:,0],L-small[:,1]))]
    uv=np.array([(i-(w-1)/2,j-(m-1)/2) for i in range(w) for j in range(m)])
    c,s=math.cos(theta),math.sin(theta)
    blocks += [uv@np.array([[c,s],[-s,c]])+L/2]
    xy=np.concatenate(blocks)
    groups=np.concatenate([np.full(len(b),g) for g,b in enumerate(blocks)])
    angles=np.zeros(len(xy)); angles[groups==len(blocks)-1]=theta
    return np.column_stack((xy,angles)),groups

def pair_gaps(p):
    """All pair SAT gaps; nonnegative means disjoint interiors."""
    i,j=np.triu_indices(len(p),1)
    d=p[j,:2]-p[i,:2]
    ai=p[i,2];aj=p[j,2]
    axes=np.stack((np.column_stack((np.cos(ai),np.sin(ai))),np.column_stack((-np.sin(ai),np.cos(ai))),np.column_stack((np.cos(aj),np.sin(aj))),np.column_stack((-np.sin(aj),np.cos(aj)))),axis=1)
    projections=np.abs(np.einsum('ik,ijk->ij',d,axes))
    delta=ai-aj
    radius=(1+np.abs(np.cos(delta))+np.abs(np.sin(delta)))/2
    gaps=projections-radius[:,None]
    return i,j,gaps,axes

def compact(k,w,r,m,theta, return_packing=False):
    reference,g=strip_blocks(k,w,r,m)
    p,_=strip_blocks(k,w,r,m,theta)
    ng=int(g.max()+1); nv=2*ng+1
    i,j,gaps,old_axes=pair_gaps(reference)
    cross=g[i]!=g[j]
    i,j,gaps,old_axes=i[cross],j[cross],gaps[cross],old_axes[cross]
    chosen=gaps.argmax(axis=1)
    # Start with local contacts. Add distant constraints only if needed.
    near=np.max(np.abs(reference[j,:2]-reference[i,:2]),axis=1)<3.0
    selected=set(np.flatnonzero(near).tolist())
    old_a=old_axes[np.arange(len(i)),chosen]
    sign=np.sign(np.sum((reference[j,:2]-reference[i,:2])*old_a,axis=1))
    sign[sign==0]=1
    def constraints(indices):
        ii=i[indices];jj=j[indices];ch=chosen[indices]
        # Axes 0,1 belong to i; axes 2,3 belong to j.
        angle=np.where(ch<2,p[ii,2],p[jj,2])+(ch%2)*math.pi/2
        ax=np.column_stack((np.cos(angle),np.sin(angle)))*sign[indices,None]
        ai=p[ii,2];aj=p[jj,2]
        radius=(np.abs(np.cos(angle-ai))+np.abs(np.sin(angle-ai))+np.abs(np.cos(angle-aj))+np.abs(np.sin(angle-aj)))/2
        a=np.zeros((len(indices),nv))
        rows=np.arange(len(indices))
        a[rows,2*g[ii]]=ax[:,0];a[rows,2*g[ii]+1]=ax[:,1]
        a[rows,2*g[jj]]-=ax[:,0];a[rows,2*g[jj]+1]-=ax[:,1]
        b=np.sum((p[jj,:2]-p[ii,:2])*ax,axis=1)-radius-GAP
        return a,b
    # Only each block's extreme corner matters for containment.
    ac=[];bc=[]
    for z in range(ng):
        pts=p[g==z];h=(abs(math.cos(pts[0,2]))+abs(math.sin(pts[0,2])))/2
        for dim in (0,1):
            row=np.zeros(nv);row[2*z+dim]=-1
            ac.append(row);bc.append(float(pts[:,dim].min()-h))
            row=np.zeros(nv);row[2*z+dim]=1;row[-1]=-1
            ac.append(row);bc.append(float(-pts[:,dim].max()-h))
    objective=np.zeros(nv);objective[-1]=1
    for _ in range(5):
        a,b=constraints(np.array(sorted(selected),dtype=int))
        fit=linprog(objective,A_ub=np.vstack((np.array(ac),a)),b_ub=np.concatenate((bc,b)),bounds=[(None,None)]*(nv-1)+[(0,None)],method='highs')
        if not fit.success:return (math.inf,None) if return_packing else math.inf
        q=p.copy();q[:,:2]+=fit.x[:-1].reshape(ng,2)[g]
        # Check the same fixed separating axes, including formerly distant pairs.
        all_a,all_b=constraints(np.arange(len(i)))
        violated=np.flatnonzero(all_a@fit.x-all_b>1e-7)
        new=set(violated.tolist())-selected
        if not new:
            if return_packing:return side(q),q
            return side(q)
        selected.update(new)
    return (math.inf,None) if return_packing else math.inf


### packing_pipeline.py

In [ ]:
%%writefile packing_pipeline.py
"""Offline atlas transfer, border expansion, and monotone subset selection."""
import math
import numpy as np
from solver import grid, normalize, side, strip_blocks, pair_gaps

def border_expand(p,t):
    p=normalize(p,0)
    L=side(p)
    k=int(math.floor(L+1e-9))
    if t==0:return p.copy()
    right=np.array([(L+i+.5,j+.5,0.) for i in range(t) for j in range(k)])
    top=np.array([(i+.5,L+j+.5,0.) for j in range(t) for i in range(k+t)])
    return np.vstack((p,right,top))

def trim_order(p,shift=(0.,0.)):
    """Keep spatially central squares; bounding side can only decrease."""
    h=(np.abs(np.cos(p[:,2]))+np.abs(np.sin(p[:,2])))/2
    low=(p[:,:2]-h[:,None]).min(axis=0)
    high=(p[:,:2]+h[:,None]).max(axis=0)
    center=(low+high)/2+np.asarray(shift)
    radius=np.max(np.abs(p[:,:2]-center)+h[:,None],axis=1)
    return np.argsort(radius,kind='stable')

def solve(reference_records, max_n=1000, use_border=True, use_strips=True, core_records=None):
    ns=np.arange(1,max_n+1)
    best=np.ceil(np.sqrt(ns)).astype(float)
    packings={n:grid(n) for n in ns}
    provenance={int(n):'grid' for n in ns}
    candidates=[]
    for rec in reference_records:
        p=np.array(rec['packing'],dtype=float)
        candidates.append((p,'reference:'+rec['url'].rsplit('/',1)[-1]))
    if core_records:
        from corner_search import corner_layout
        for idx,rec in enumerate(core_records):
            p=corner_layout(rec['L'],rec['w'],rec['m'],rec['sx'],rec['sy'])
            candidates.append((normalize(p),f'core:{idx}'))
    if use_strips:
        for k in range(2,33):
            for w in range(1,7):
                for r in range(math.floor(1+w/math.sqrt(2))+1):
                    L=k+w/math.sqrt(2)
                    m=math.floor(math.sqrt(2)*k+1e-10) if r==0 else min(math.floor(math.sqrt(2)*k+1e-10),math.floor(math.sqrt(2)*(L-r-1)+1e-10))
                    N=k*(k-1)+r*(r+1)+w*m
                    if m<=0 or N>max_n+80 or L>=math.ceil(math.sqrt(max(1,N)))-1e-7:continue
                    p,_=strip_blocks(k,w,r,m)
                    # Analytic construction requires disjoint corner triangles.
                    if r>1+w/math.sqrt(2)+1e-10:continue
                    p=normalize(p)
                    if pair_gaps(p)[2].max(axis=1).min() < -1e-9:continue
                    candidates.append((p,f'strip:k{k},w{w},r{r},m{m}'))
    stages={}
    for p,label in candidates:
        L=side(p)
        max_t=max(0,int(math.ceil(math.sqrt(max_n))-L)) if use_border else 0
        for t in range(max_t+1):
            q=border_expand(p,t)
            if len(q)==0:continue
            Lq=side(q)
            # A layout of side L cannot improve n <= floor(L)^2.
            first=1
            last=min(max_n,len(q))
            if first>last:continue
            for dx in (-.5,0.,.5):
                for dy in (-.5,0.,.5):
                    order=trim_order(q,(dx,dy))
                    sorted_q=q[order]
                    h=(np.abs(np.cos(sorted_q[:,2]))+np.abs(np.sin(sorted_q[:,2])))/2
                    lower=np.minimum.accumulate(sorted_q[:,:2]-h[:,None],axis=0)
                    upper=np.maximum.accumulate(sorted_q[:,:2]+h[:,None],axis=0)
                    lengths=np.max(upper-lower,axis=1)
                    improved=np.flatnonzero(lengths[:last]<best[:last]-1e-8)
                    for ix in improved:
                        n=int(ix+1)
                        best[ix]=lengths[ix]
                        packings[n]=sorted_q[:n].copy()
                        provenance[n]=label+(f'+border{t}' if t else '')
    return best,packings,provenance


### hybrid_search.py

In [ ]:
%%writefile hybrid_search.py
"""Combine credited portfolios, CUDA families, and independently measured crops."""
import csv,json,math,time
from pathlib import Path
from collections import defaultdict
import numpy as np
from solver import grid,side,normalize
from packing_pipeline import border_expand,trim_order
from gpu_families import construct

def read_csv(path):
    groups=defaultdict(list)
    with Path(path).open(newline='') as f:
        for row in csv.DictReader(f):groups[int(row['n'])].append([float(row[k]) for k in ['x','y','angle']])
    return {n:np.asarray(p) for n,p in groups.items()}

def export(path,bank,margin=4e-7):
    with Path(path).open('w',newline='') as f:
        w=csv.writer(f);w.writerow(['id','n','square_id','x','y','angle']);idx=0
        for n,p in sorted(bank.items()):
            p=normalize(p,margin)
            if side(p)>=math.ceil(math.sqrt(n))-1e-8:continue
            for j,(x,y,a) in enumerate(p):
                w.writerow([idx,n,j,format(x,'.17g'),format(y,'.17g'),format(a,'.17g')]);idx+=1

def solve(include_gpu=True,transfer=True,include_recent=True):
    start=time.perf_counter();bank={n:grid(n) for n in range(1,1001)};lengths=np.ceil(np.sqrt(np.arange(1,1001)))
    labels={n:'grid' for n in bank};ablation=[]
    def checkpoint(stage):
        row=dict(stage=stage,score=float(np.sum(lengths/np.sqrt(np.arange(1,1001))-1)))
        ablation.append(row);print(json.dumps(row),flush=True)
    def offer(p,label,crops=False):
        if not crops:
            n=len(p)
            if n<=1000 and side(p)<lengths[n-1]-1e-8:
                bank[n]=p.copy();lengths[n-1]=side(p);labels[n]=label
            return
        for dx in (-.5,0,.5):
            for dy in (-.5,0,.5):
                q=p[trim_order(p,(dx,dy))];h=(np.abs(np.cos(q[:,2]))+np.abs(np.sin(q[:,2])))/2
                lo=np.minimum.accumulate(q[:,:2]-h[:,None],axis=0);hi=np.maximum.accumulate(q[:,:2]+h[:,None],axis=0)
                ll=np.max(hi-lo,axis=1)[:1000]
                for j in np.flatnonzero(ll<lengths[:len(ll)]-1e-8):
                    n=int(j+1);bank[n]=q[:n].copy();lengths[j]=ll[j];labels[n]=label
    avazaku=read_csv('artifacts/avazaku-v1/submission.csv')
    for p in avazaku.values():offer(p,'avazaku v1')
    checkpoint('Squarecraft v1')
    own=read_csv('artifacts/submission.csv')
    for p in own.values():offer(p,'our v1')
    checkpoint('Best of both portfolios')
    candidates=[(p,'avazaku transferred') for p in avazaku.values()]+[(p,'our v1 transferred') for p in own.values()]
    if include_recent:
        references=json.loads(Path('artifacts/recent_references.json').read_text())
        for rec in references:
            p=np.array(rec['packing'],dtype=float)
            label=f"recent {rec['source']['group']} n={rec['N']}"
            offer(p,label)
            if side(p)<math.ceil(math.sqrt(len(p)))-1e-7:candidates.append((p,label))
        checkpoint('Recent credited numerical witnesses')
    np.savez_compressed('artifacts/hybrid_baseline.npz',**{str(n):p for n,p in bank.items()})
    if transfer:
        for p,label in candidates:
            for t in range(max(0,int(32-side(p)))+1):offer(border_expand(p,t),label,True)
        checkpoint('Cross-size transfer and crops')
    if include_gpu:
        recipes=json.loads(Path('artifacts/gpu_family_recipes.json').read_text())
        for rec in recipes:
            p=construct(rec['corners'],rec['b'],rec['c'],math.radians(rec['degrees']))
            label=f"CUDA family {rec['corners']}/{rec['b']}/{rec['c']}/{rec['degrees']}deg"
            for t in range(max(0,int(32-side(p)))+1):offer(border_expand(p,t),label,True)
        checkpoint('CUDA generalized cores + crops')
    np.savez_compressed('artifacts/hybrid_packings.npz',**{str(n):p for n,p in bank.items()})
    Path('artifacts/hybrid_provenance.json').write_text(json.dumps(labels,indent=2))
    Path('artifacts/hybrid_ablation.json').write_text(json.dumps(ablation,indent=2))
    export('artifacts/hybrid_submission.csv',bank)
    print('DONE seconds',time.perf_counter()-start,flush=True)
    return bank,ablation

if __name__=='__main__':solve()


### gpu_families.py

In [ ]:
%%writefile gpu_families.py
"""CUDA enumeration of four staircases and an arbitrarily rotated rectangular core.

Starting point: avazaku's Squarecraft v1, asymmetric 45-degree cores.
Extension: independent corner lengths and a free core angle. The minimum
container follows from SAT between two rectangles of feasible core centers.
"""
import argparse
import json
import math
import time
from pathlib import Path
import numpy as np
import torch
from scipy.spatial import cKDTree
from solver import side, normalize


def corner_parameters(limit=33):
    # Rotate a largest corner to BL; reflect to restrict the core to [0,45] deg.
    return np.array([(bl,br,tr,tl) for bl in range(1,limit)
                     for br in range(min(bl,limit-bl)+1)
                     for tr in range(bl+1)
                     for tl in range(min(bl,limit-bl)+1)], dtype=np.int16)


def minimum_side(corners,b,c,theta,xp=np):
    co,si=math.cos(theta),math.sin(theta)
    bl,br,tr,tl=[corners[...,k] for k in range(4)]
    # Support of a nonempty staircase on the inward core normal.
    # For an empty staircase omit its plane using the no-corner limit -infinity.
    # Zero-step support = min(cos,sin) is conservative and valid.
    t_bl=bl*max(co,si)+min(co,si)
    t_br=br*max(co,si)+min(co,si)
    t_tr=tr*max(co,si)+min(co,si)
    t_tl=tl*max(co,si)+min(co,si)
    hx=(b*co+c*si)/2; hy=(b*si+c*co)/2
    B=b+t_bl+t_tr; C=c+t_br+t_tl
    du=(t_bl-t_tr)/2; dv=(t_br-t_tl)/2
    terms=[2*hx,2*hy,B/(co+si),C/(co+si),
           (xp.abs(du)+B/2+co*hx+si*hy)/(co+si),
           (xp.abs(dv)+C/2+si*hx+co*hy)/(co+si),
           (2*xp.abs(co*du-si*dv)+2*hx+co*B+si*C)/(1+(co+si)**2),
           (2*xp.abs(si*du+co*dv)+2*hy+si*B+co*C)/(1+(co+si)**2),
           xp.maximum(xp.maximum(bl+br,br+tr),xp.maximum(tr+tl,tl+bl))]
    result=terms[0]
    for term in terms[1:]:result=xp.maximum(result,term)
    return result


def clip(poly,a,b,rhs):
    out=[]
    for p,q in zip(poly,poly[1:]+poly[:1]):
        fp=a*p[0]+b*p[1]-rhs; fq=a*q[0]+b*q[1]-rhs
        if fp<=1e-10:out.append(p)
        if (fp<0)!=(fq<0):
            t=fp/(fp-fq);out.append([p[0]+t*(q[0]-p[0]),p[1]+t*(q[1]-p[1])])
    return out


def construct(corners,b,c,theta):
    corners=np.asarray(corners,dtype=float)
    L=float(minimum_side(corners,b,c,theta))+2e-8
    co,si=math.cos(theta),math.sin(theta); hx=(b*co+c*si)/2;hy=(b*si+c*co)/2
    tb=corners*max(co,si)+min(co,si);bl,br,tr,tl=tb
    poly=[[hx,hy],[L-hx,hy],[L-hx,L-hy],[hx,L-hy]]
    for a,z,r in [(co,si,L*(co+si)-b/2-tr),(-co,-si,-b/2-bl),
                  (-si,co,co*L-c/2-tl),(si,-co,si*L-c/2-br)]:
        poly=clip(poly,a,z,r)
        if not poly:raise ValueError(('empty center domain',corners,b,c,theta,L))
    cx,cy=np.mean(poly,axis=0)
    poses=[]
    for k,steps in enumerate(corners.astype(int)):
        for i in range(steps):
            for j in range(steps-i):
                x,y=i+.5,j+.5
                for _ in range(k):x,y=L-y,x
                poses.append([x,y,0.])
    for i in range(b):
        for j in range(c):
            u=i-(b-1)/2;v=j-(c-1)/2
            poses.append([cx+co*u-si*v,cy+si*u+co*v,theta])
    return normalize(np.asarray(poses),4e-7)


def min_gap(p):
    pairs=np.asarray(list(cKDTree(p[:,:2]).query_pairs(1.5)),dtype=int).reshape(-1,2)
    if not len(pairs):return float('inf')
    i,j=pairs.T;ai,aj=p[i,2],p[j,2];d=p[j,:2]-p[i,:2]
    proj=np.stack([np.abs(d[:,0]*np.cos(t)+d[:,1]*np.sin(t)) for t in [ai,ai+np.pi/2,aj,aj+np.pi/2]],axis=1)
    r=(1+np.abs(np.cos(ai-aj))+np.abs(np.sin(ai-aj)))/2
    return float((proj.max(axis=1)-r).min())


def search(degrees,chunk=256,max_n=1100,limit=33,output='artifacts/gpu_family_recipes.json'):
    assert torch.cuda.is_available(),'CUDA is required for this enumeration'
    started=time.perf_counter();params=corner_parameters(limit)
    counts=(params.astype(np.int64)*(params+1)).sum(axis=1)//2
    params=params[counts<=max_n];counts=counts[counts<=max_n]
    bc=np.array([(b,c) for b in range(1,46) for c in range(1,46)],dtype=np.int16)
    bc=bc[np.prod(bc.astype(int),axis=1)<=max_n]
    b=torch.tensor(bc[:,0],dtype=torch.float32,device='cuda')[None]
    c=torch.tensor(bc[:,1],dtype=torch.float32,device='cuda')[None]
    capacity_core=(b*c).long()
    best=torch.full((max_n+1,),float('inf'),device='cuda');winning=torch.full((max_n+1,),-1,device='cuda',dtype=torch.long)
    evaluated=0;rows=[]
    for angle_index,deg in enumerate(degrees):
        theta=math.radians(deg);at=time.perf_counter()
        for start in range(0,len(params),chunk):
            pp=torch.tensor(params[start:start+chunk],dtype=torch.float32,device='cuda')[:,None,:]
            ns=torch.tensor(counts[start:start+chunk],device='cuda')[:,None]+capacity_core
            L=minimum_side(pp,b,c,theta,torch)
            valid=(ns<=max_n)&(L<torch.ceil(torch.sqrt(ns.float()))-1e-5)&(L<32)
            idx=ns.clamp_max(max_n).ravel()
            values=torch.where(valid,L,float('inf')).ravel()
            newbest=best.clone();newbest.scatter_reduce_(0,idx,values,reduce='amin',include_self=True)
            mask=(values==newbest[idx])&(values<best[idx])
            # Stable tie-breaking: smallest flattened parameter id.
            ids=(angle_index*len(params)+(start+torch.arange(len(pp),device='cuda'))[:,None])*len(bc)+torch.arange(len(bc),device='cuda')[None,:]
            chosen=torch.full_like(winning,torch.iinfo(torch.long).max)
            chosen.scatter_reduce_(0,idx,torch.where(mask,ids.ravel(),torch.iinfo(torch.long).max),reduce='amin',include_self=True)
            changed=newbest<best;winning[changed]=chosen[changed];best=newbest
            evaluated+=len(pp)*len(bc)
        torch.cuda.synchronize()
        row={'degrees':deg,'seconds':time.perf_counter()-at,'capacities':int(torch.isfinite(best).sum())}
        rows.append(row);print(json.dumps(row),flush=True)
        ids=winning.cpu().numpy();out=[]
        for n,z in enumerate(ids):
            if z<0:continue
            ai,rest=divmod(int(z),len(params)*len(bc));pi,bi=divmod(rest,len(bc))
            bb,cc=map(int,bc[bi]);corners=list(map(int,params[pi]));angle=math.radians(degrees[ai])
            out.append(dict(n=n,corners=corners,b=bb,c=cc,degrees=degrees[ai],L=float(minimum_side(np.array(corners),bb,cc,angle))))
        Path(output).write_text(json.dumps(out,indent=2))
        Path('research/gpu_family_status.json').write_text(json.dumps(dict(gpu=torch.cuda.get_device_name(0),evaluated=evaluated,elapsed=time.perf_counter()-started,results=rows),indent=2))
    accepted=[];bad=[]
    for rec in out:
        p=construct(rec['corners'],rec['b'],rec['c'],math.radians(rec['degrees']))
        gap=min_gap(p)
        if gap<1e-8:bad.append((rec['n'],gap))
        else:accepted.append(rec)
    assert not bad,bad[:10]
    print('DONE',json.dumps(dict(evaluated=evaluated,seconds=time.perf_counter()-started,validated=len(accepted))),flush=True)
    return accepted


if __name__=='__main__':
    ap=argparse.ArgumentParser();ap.add_argument('--degrees',nargs='+',type=float,default=[45,44,43,42,40,37.5,35,30,25,20,15,10,5])
    ap.add_argument('--chunk',type=int,default=256);ap.add_argument('--limit',type=int,default=33)
    args=ap.parse_args();search(args.degrees,chunk=args.chunk,limit=args.limit)


### gpu_topology.py

In [ ]:
%%writefile gpu_topology.py
"""Independent CUDA basin hopping: spatial hashing + per-island L-BFGS.

Every square has independent x/y/angle. Overlaps are allowed during repair;
incumbents are admitted only after independent full SAT/interval checks.
"""
import argparse,json,math,time
from pathlib import Path
import numpy as np
import cupy as cp
from scipy.spatial import cKDTree
from solver import side,normalize,grid
from gpu_families import min_gap
from hybrid_search import export

CUDA=r'''
extern "C" {
__device__ double sg(double x){return (x>1e-12)-(x < -1e-12);}
__device__ double dot(const double*a,const double*b,int m,double*r){
 double z=0;for(int k=threadIdx.x;k<m;k+=blockDim.x)z+=a[k]*b[k];
 r[threadIdx.x]=z;__syncthreads();
 for(int t=blockDim.x/2;t;t/=2){if(threadIdx.x<t)r[threadIdx.x]+=r[threadIdx.x+t];__syncthreads();}
 double answer=r[0];__syncthreads();return answer;
}
__device__ double objective(double*q,double*g,int n,double L,int nc,int*head,int*next,double*cs,double*r,double*maxpen){
 int tid=threadIdx.x;
 for(int k=tid;k<nc*nc;k+=blockDim.x)head[k]=-1;
 __syncthreads();
 for(int i=tid;i<n;i+=blockDim.x){
  cs[2*i]=cos(q[3*i+2]);cs[2*i+1]=sin(q[3*i+2]);
  int x=max(0,min(nc-1,int(floor((q[3*i]+3)/1.5))));
  int y=max(0,min(nc-1,int(floor((q[3*i+1]+3)/1.5))));
  next[i]=atomicExch(head+y*nc+x,i);
 }__syncthreads();
 double energy=0,mp=0;
 for(int i=tid;i<n;i+=blockDim.x){
  double x=q[3*i],y=q[3*i+1],c=cs[2*i],s=cs[2*i+1];
  double gx=0,gy=0,ga=0,h=.5*(fabs(c)+fabs(s)),dh=.5*(-s*sg(c)+c*sg(s));
  double b[4]={h-x,h-y,x+h-L,y+h-L};
  for(int k=0;k<4;k++)if(b[k]>0){double p=b[k];energy+=2*p*p;mp=fmax(mp,p);ga+=4*p*dh;
   if(k==0)gx-=4*p;if(k==1)gy-=4*p;if(k==2)gx+=4*p;if(k==3)gy+=4*p;}
  int cx=max(0,min(nc-1,int(floor((x+3)/1.5)))),cy=max(0,min(nc-1,int(floor((y+3)/1.5))));
  for(int yy=max(0,cy-1);yy<=min(nc-1,cy+1);yy++)for(int xx=max(0,cx-1);xx<=min(nc-1,cx+1);xx++)
   for(int j=head[yy*nc+xx];j>=0;j=next[j])if(j!=i){
    double dx=q[3*j]-x,dy=q[3*j+1]-y;if(dx*dx+dy*dy>2.00000001)continue;
    double cj=cs[2*j],sj=cs[2*j+1],cd=c*cj+s*sj,sd=c*sj-s*cj;
    double rad=.5*(1+fabs(cd)+fabs(sd))+2e-10,rd=.5*(-sd*sg(cd)+cd*sg(sd));
    double ux[4]={c,-s,cj,-sj},uy[4]={s,c,sj,cj};int a=0;double z=dx*ux[0]+dy*uy[0];
    for(int k=1;k<4;k++){double v=dx*ux[k]+dy*uy[k];if(fabs(v)>fabs(z)){a=k;z=v;}}
    double p=rad-fabs(z);if(p<=0)continue;
    energy+=.25*p*p;mp=fmax(mp,p);double sign=sg(z);
    gx+=p*sign*ux[a];gy+=p*sign*uy[a];
    ga-=p*(rd+(a<2?sign*(-dx*uy[a]+dy*ux[a]):0));
   }
  g[3*i]=gx;g[3*i+1]=gy;g[3*i+2]=ga;
 }
 r[tid]=energy;r[blockDim.x+tid]=mp;__syncthreads();
 for(int t=blockDim.x/2;t;t/=2){if(tid<t){r[tid]+=r[tid+t];r[blockDim.x+tid]=fmax(r[blockDim.x+tid],r[blockDim.x+tid+t]);}__syncthreads();}
 if(tid==0)*maxpen=r[blockDim.x];double answer=r[0];__syncthreads();return answer;
}
__global__ void eval(double*q,double*g,double*e,int n,int nc,double*Ls,int*head,int*next,double*cs){
 __shared__ double r[512],mp;
 int b=blockIdx.x;double v=objective(q+b*3*n,g+b*3*n,n,Ls[b],nc,head+b*nc*nc,next+b*n,cs+b*2*n,r,&mp);
 if(threadIdx.x==0){e[2*b]=v;e[2*b+1]=mp;}
}
__global__ void lbfgs(double*Q,double*work,double*history,double*out,int n,int nc,double*Ls,int*heads,int*links,double*trigs,int steps,int M){
 __shared__ double r[512],mp,alpha[12],rho[12],syarr[12],yyarr[12];
 int b=blockIdx.x,tid=threadIdx.x,d=3*n;
 double*q=Q+b*d,*g=work+b*d*5,*v=g+d,*trial=v+d,*gt=trial+d,*direction=gt+d;
 double*S=history+b*2*M*d,*Y=S+M*d;int*head=heads+b*nc*nc,*next=links+b*n;double*cs=trigs+b*2*n;
 double L=Ls[b],E=objective(q,g,n,L,nc,head,next,cs,r,&mp);int stored=0,last=0,its=0;
 for(int it=0;it<steps;it++){
  its=it;if(mp<2e-11)break;
  for(int k=tid;k<d;k+=blockDim.x)v[k]=g[k];__syncthreads();
  for(int j=0;j<stored;j++){int h=(last-1-j+M)%M;double z=dot(S+h*d,v,d,r)*rho[h];
   if(tid==0)alpha[h]=z;__syncthreads();for(int k=tid;k<d;k+=blockDim.x)v[k]-=z*Y[h*d+k];__syncthreads();}
  double scale=stored?fmin(10.,fmax(1e-6,syarr[(last-1+M)%M]/yyarr[(last-1+M)%M])):.2;
  for(int k=tid;k<d;k+=blockDim.x)v[k]*=scale;__syncthreads();
  for(int j=stored-1;j>=0;j--){int h=(last-1-j+M)%M;double z=dot(Y+h*d,v,d,r)*rho[h];
   double a=alpha[h]-z;for(int k=tid;k<d;k+=blockDim.x)v[k]+=a*S[h*d+k];__syncthreads();}
  for(int k=tid;k<d;k+=blockDim.x)direction[k]=-v[k];__syncthreads();
  double slope=dot(g,direction,d,r);
  if(slope>=-1e-30){for(int k=tid;k<d;k+=blockDim.x)direction[k]=-.2*g[k];__syncthreads();slope=dot(g,direction,d,r);stored=0;}
  double dm=0;for(int k=tid;k<d;k+=blockDim.x)dm=fmax(dm,fabs(direction[k]));r[tid]=dm;__syncthreads();
  for(int t=blockDim.x/2;t;t/=2){if(tid<t)r[tid]=fmax(r[tid],r[tid+t]);__syncthreads();}
  double step=fmin(1.,.2/fmax(1e-30,r[0])),En=E;bool ok=false;
  for(int ls=0;ls<24;ls++){
   for(int k=tid;k<d;k+=blockDim.x)trial[k]=q[k]+step*direction[k];__syncthreads();
   En=objective(trial,gt,n,L,nc,head,next,cs,r,&mp);
   if(En<=E+1e-4*step*slope){ok=true;break;}step*=.5;
  }
  if(!ok)break;
  for(int k=tid;k<d;k+=blockDim.x){v[k]=trial[k]-q[k];direction[k]=gt[k]-g[k];}__syncthreads();
  double sy=dot(v,direction,d,r),yy=dot(direction,direction,d,r);
  if(sy>1e-24&&yy>1e-30){for(int k=tid;k<d;k+=blockDim.x){S[last*d+k]=v[k];Y[last*d+k]=direction[k];}__syncthreads();
   if(tid==0){rho[last]=1/sy;syarr[last]=sy;yyarr[last]=yy;}__syncthreads();last=(last+1)%M;stored=min(M,stored+1);}
  for(int k=tid;k<d;k+=blockDim.x){q[k]=trial[k];g[k]=gt[k];}__syncthreads();E=En;
 }
 if(tid==0){out[3*b]=E;out[3*b+1]=mp;out[3*b+2]=its;}
}
}
'''
mod=cp.RawModule(code=CUDA,options=('--std=c++17',),name_expressions=['eval','lbfgs'])

def buffers(q,L):
    q=cp.asarray(q,dtype=cp.float64);B,n,_=q.shape;nc=int(math.ceil((max(L)+6)/1.5))+1
    return q,n,nc,cp.asarray(L,dtype=cp.float64),cp.empty((B,nc*nc),cp.int32),cp.empty((B,n),cp.int32),cp.empty((B,n,2),cp.float64)

def energy(q,L):
    q,n,nc,L,h,l,cs=buffers(q,L);g=cp.empty_like(q);e=cp.empty((len(q),2),cp.float64)
    mod.get_function('eval')((len(q),),(256,),(q,g,e,np.int32(n),np.int32(nc),L,h,l,cs))
    return cp.asnumpy(e),cp.asnumpy(g)

def quench(q,L,steps=1200):
    q,n,nc,L,h,l,cs=buffers(q,L);B=len(q);M=8
    work=cp.empty((B,5,n,3),cp.float64);hist=cp.empty((B,2,M,n,3),cp.float64);out=cp.empty((B,3),cp.float64)
    mod.get_function('lbfgs')((B,),(256,),(q,work,hist,out,np.int32(n),np.int32(nc),L,h,l,cs,np.int32(steps),np.int32(M)))
    return cp.asnumpy(q),cp.asnumpy(out)

def check():
    rng=np.random.default_rng(17);q=rng.uniform(.4,2.5,(1,5,3));q[:,:,2]*=.2
    e,g=energy(q,[3.]);errs=[]
    for i,k in [(0,0),(1,2),(3,1),(4,2)]:
        plus=q.copy();minus=q.copy();plus[0,i,k]+=1e-6;minus[0,i,k]-=1e-6
        fd=(energy(plus,[3.])[0][0,0]-energy(minus,[3.])[0][0,0])/2e-6
        errs.append(abs(fd-g[0,i,k]))
    assert max(errs)<1e-6,errs
    qq,o=quench(q,[3.],100)
    assert o[0,0]<e[0,0],(e,o)
    print('CUDA analytic gradient / descent checks passed',max(errs),flush=True)

def reinsert(p,rng,count):
    """Destroy a spatial region; insert into sampled largest holes, new angles."""
    n=len(p);L=side(p);pivot=p[rng.integers(n),:2]
    ids=np.argsort(np.sum((p[:,:2]-pivot)**2,axis=1))[:count]
    keep=np.ones(n,bool);keep[ids]=False;xy=p[keep,:2].copy()
    for i in ids:
        cand=rng.uniform(.6,max(.6001,L-.6),(256,2));dist=cKDTree(xy).query(cand)[0] if len(xy) else np.ones(256)
        pos=cand[np.argmax(dist)];p[i,:2]=pos;p[i,2]=rng.choice([0,np.pi/4,rng.uniform(-np.pi/4,np.pi/4)])
        xy=np.vstack((xy,pos))
    return p

def starts(n,bank,refs,rng,B,roundno,archive=None):
    base=normalize(bank[n],0);L=side(base);seeds=[base]
    seeds.extend(normalize(np.array(r['packing']),0) for r in refs if r['N']==n and r['L']<L*1.04)
    if n%4==0 and n//4 in bank:
        old=normalize(bank[n//4],0);parts=[]
        for x,y,a in old:
            co,si=np.cos(a),np.sin(a);rot=np.array([[co,si],[-si,co]])
            for off in [[-.5,-.5],[.5,-.5],[-.5,.5],[.5,.5]]:parts.append([*(np.array(off)@rot+2*np.array([x,y])),a])
        seeds.append(normalize(np.array(parts),0))
    if archive:seeds.extend(archive)
    seeds.append(normalize(grid(n),0))
    qs=[];kinds=[]
    for b in range(B):
        q=seeds[b%len(seeds)].copy();q[:,:2]*=(L-1)/(max(1e-9,side(q)-1))
        kind=b%6
        if b<2:kind=0
        if kind==0:q[:,:2]+=rng.normal(0,.0003,q[:,:2].shape);q[:,2]+=rng.normal(0,.0002,n)
        elif kind==1:
            ids=np.argsort(np.sum((q[:,:2]-q[rng.integers(n),:2])**2,axis=1))[:max(3,n//8)]
            a=rng.uniform(-.4,.4);c,s=np.cos(a),np.sin(a);xy=q[ids,:2].mean(0)
            q[ids,:2]=(q[ids,:2]-xy)@np.array([[c,s],[-s,c]])+xy+rng.normal(0,.2,2);q[ids,2]+=a
        elif kind==2:q=reinsert(q,rng,max(2,n//12))
        elif kind==3:
            q[:,:2]+=rng.normal(0,.05,q[:,:2].shape);q[:,2]+=rng.normal(0,.08,n)
        elif kind==4:
            cut=rng.uniform(.2,.8)*L;ids=q[:,1]>cut;q[ids,0]+=rng.choice([-.35,.35]);q[ids,2]+=rng.uniform(-.1,.1)
        else:
            # Crossover: retain one half, replace the other by a different seed.
            other=seeds[(b+1)%len(seeds)];ids=np.argsort(q[:,0]);other=other[np.argsort(other[:,0])]
            q[ids[n//2:]]=other[n//2:];q[:,0]+=rng.normal(0,.02,n)
        qs.append(q);kinds.append(kind)
    return np.array(qs),kinds

def repair_tiny(p):
    """Pair-specific clearance correction; reject large expansions."""
    p=normalize(p,0);pairs=cKDTree(p[:,:2]).query_pairs(1.415,output_type='ndarray')
    if not len(pairs):return p
    i,j=pairs.T;d=p[j,:2]-p[i,:2];a=p[i,2];b=p[j,2]
    z=np.maximum.reduce([np.abs(d[:,0]*np.cos(a)+d[:,1]*np.sin(a)),np.abs(-d[:,0]*np.sin(a)+d[:,1]*np.cos(a)),np.abs(d[:,0]*np.cos(b)+d[:,1]*np.sin(b)),np.abs(-d[:,0]*np.sin(b)+d[:,1]*np.cos(b))])
    rad=.5*(1+np.abs(np.cos(a-b))+np.abs(np.sin(a-b)))
    factor=max(1.,float(np.max((rad+2e-10)/np.maximum(z,1e-30))))
    if factor>1.00002:return None
    p[:,:2]*=factor;return normalize(p,0)

def search(seconds=240,B=96,steps=1600,ns=None,input_path='artifacts/gpu_deep_packings.npz',tag='gpu_topology',staged=False):
    check();loaded=np.load(input_path);bank={int(n):loaded[n] for n in loaded.files}
    refs=json.loads(Path('artifacts/recent_references.json').read_text());rng=np.random.default_rng(20261022)
    if ns is None:ns=[17,27,34,51,68,80,109,126,143,167,211,241,273,307,500,645,646,804]
    start=time.perf_counter();rows=[];hits=[];roundno=0;archives={}
    while time.perf_counter()-start<seconds:
        n=ns[roundno%len(ns)];old=side(bank[n]);q,kinds=starts(n,bank,refs,rng,B,roundno,archives.get(n))
        # Different requested contractions prevent all islands following one basin.
        target=old-np.geomspace(2e-7,.025,B);at=time.perf_counter()
        archived=0
        if staged:
            # Make room to cross contact barriers, retaining several alternative basins.
            direct=q[:2].copy()
            loose=old+np.linspace(.015,.25,B)
            q,soft=quench(q,loose,max(500,steps//2))
            candidates=[];signatures=set()
            for b in np.argsort(soft[:,0]):
                if soft[b,1]>1e-6:continue
                p=repair_tiny(q[b])
                if p is None or side(p)>old+.3:continue
                # Coarse spatial/orientation occupancy removes near-duplicate islands.
                signature=tuple(np.histogramdd(np.column_stack((p[:,:2]/old,p[:,2]%(np.pi/2))),bins=(6,6,4),range=((0,1),(0,1),(0,np.pi/2)))[0].astype(int).ravel())
                if signature in signatures:continue
                signatures.add(signature);candidates.append(p)
                if len(candidates)>=8:break
            if candidates:archives[n]=candidates
            archived=len(candidates);q[:2]=direct
            q,_=quench(q,np.full(B,old+.003),max(400,steps//2))
        q,stats=quench(q,target,steps);feasible=0
        for b in np.argsort(stats[:,0]):
            if stats[b,1]>1e-5:continue
            p=repair_tiny(q[b])
            if p is not None and side(p)<side(bank[n])-5e-8 and min_gap(p)>-2e-10:
                feasible+=1;before=side(bank[n]);bank[n]=p;hits.append(dict(n=n,before=before,after=side(p),mutation=kinds[b],island=int(b),round=roundno))
        row=dict(round=roundno,n=n,seconds=time.perf_counter()-at,islands=B,before=old,after=side(bank[n]),best_penetration=float(stats[:,1].min()),minimum_energy=float(stats[:,0].min()),mean_iterations=float(stats[:,2].mean()),improvements=feasible,diverse_loose_basins=archived)
        rows.append(row);roundno+=1
        np.savez_compressed(f'artifacts/{tag}_packings.npz',**{str(n):p for n,p in bank.items()})
        Path(f'research/{tag}_status.json').write_text(json.dumps(dict(seconds=time.perf_counter()-start,algorithm='Spatial hash, independent per-island L-BFGS and destroy/reinsert/block/crossover starts',staged=staged,rows=rows,hits=hits),indent=2))
        print(json.dumps(row),flush=True)
    export(f'artifacts/{tag}_submission.csv',bank,margin=4e-9)
    return bank,hits

if __name__=='__main__':
    a=argparse.ArgumentParser();a.add_argument('--seconds',type=float,default=240);a.add_argument('--islands',type=int,default=96);a.add_argument('--steps',type=int,default=1600);a.add_argument('--check',action='store_true');a.add_argument('--ns',type=int,nargs='*');a.add_argument('--input',default='artifacts/gpu_deep_packings.npz');a.add_argument('--tag',default='gpu_topology');a.add_argument('--staged',action='store_true');args=a.parse_args()
    if args.check:check()
    else:search(args.seconds,args.islands,args.steps,args.ns,args.input,args.tag,args.staged)


### gpu_rect_optimizer.py

In [ ]:
%%writefile gpu_rect_optimizer.py
"""CUDA rectangular unit-square L-BFGS, with optional smooth SAT continuation.

Extends the existing independently implemented square-container kernel.
The shape remains an exact unit square; only its centre/angle may change.
Smoothing is used in exploration, never in final feasibility admission.
"""
import re
import cupy as cp
import numpy as np
import gpu_topology as square

src=square.CUDA.replace('alpha[','history_alpha[')
src=src.replace('double L,int nc','double W,double H,double tau,int nc')
src=src.replace('double*Ls,int*head','double*Ls,double*Ts,int*head')
src=src.replace('n,Ls[b],nc,','n,Ls[2*b],Ls[2*b+1],Ts[b],nc,')
src=src.replace('double L=Ls[b],E=','double W=Ls[2*b],H=Ls[2*b+1],tau=Ts[b],E=')
src=src.replace('n,L,nc,','n,W,H,tau,nc,')
src=src.replace('x+h-L,y+h-L','x+h-W,y+h-H')
# Preserve zero contact residual at exact zero for each smoothed absolute value.
src=src.replace('__device__ double sg(double x)', '__device__ double sg(double x)')
src=src.replace('double rad=.5*(1+fabs(cd)+fabs(sd))+2e-10,rd=.5*(-sd*sg(cd)+cd*sg(sd));',
'''double ac=tau>0?sqrt(cd*cd+tau*tau)-tau:fabs(cd),as=tau>0?sqrt(sd*sd+tau*tau)-tau:fabs(sd);
    double sc=tau>0?cd/sqrt(cd*cd+tau*tau):sg(cd),ss=tau>0?sd/sqrt(sd*sd+tau*tau):sg(sd);
    double rad=.5*(1+ac+as)+2e-10,rd=.5*(-sd*sc+cd*ss);''')
src=src.replace('double p=rad-fabs(z);','double az=tau>0?sqrt(z*z+tau*tau)-tau:fabs(z);double p=rad-az;')
src=src.replace('double sign=sg(z);','double sign=tau>0?z/sqrt(z*z+tau*tau):sg(z);')
mod=cp.RawModule(code=src,options=('--std=c++17',),name_expressions=['eval','lbfgs'])

def buffers(q,boxes,tau):
    q=cp.asarray(q,dtype=cp.float64);B,n,_=q.shape;boxes=np.asarray(boxes,dtype=float).reshape(B,2);nc=int(np.ceil((boxes.max()+6)/1.5))+1
    return q,B,n,nc,cp.asarray(boxes),cp.full(B,tau,cp.float64),cp.empty((B,nc*nc),cp.int32),cp.empty((B,n),cp.int32),cp.empty((B,n,2),cp.float64)

def energy(q,boxes,tau=0.):
    q,B,n,nc,ll,ts,h,l,cs=buffers(q,boxes,tau);g=cp.empty_like(q);out=cp.empty((B,2),cp.float64)
    mod.get_function('eval')((B,),(256,),(q,g,out,np.int32(n),np.int32(nc),ll,ts,h,l,cs));return cp.asnumpy(out),cp.asnumpy(g)

def quench(q,boxes,steps=1200,tau=0.):
    q,B,n,nc,ll,ts,h,l,cs=buffers(q,boxes,tau);M=8;work=cp.empty((B,5,n,3),cp.float64);hist=cp.empty((B,2,M,n,3),cp.float64);out=cp.empty((B,3),cp.float64)
    mod.get_function('lbfgs')((B,),(256,),(q,work,hist,out,np.int32(n),np.int32(nc),ll,ts,h,l,cs,np.int32(steps),np.int32(M)));return q,out

def check():
    rng=np.random.default_rng(230);q=rng.uniform(.4,2.8,(3,11,3));boxes=np.array([[3.3,2.9],[3.,3.],[4.,3.5]]);err=[]
    for tau in [0.,.002]:
        e,g=energy(q,boxes,tau)
        for i,k in [(0,0),(2,1),(7,2)]:
            p=q.copy();m=q.copy();p[0,i,k]+=1e-6;m[0,i,k]-=1e-6;fd=(energy(p,boxes,tau)[0][0,0]-energy(m,boxes,tau)[0][0,0])/2e-6;err.append(abs(fd-g[0,i,k]))
    e,g=energy(q[1:2],boxes[1:2],0);ee,gg=square.energy(q[1:2],[3.]);assert np.max(abs(ee-e))<1e-10 and np.max(abs(gg-g))<1e-10
    assert max(err)<2e-6,err
    return dict(max_gradient_error=max(err),square_endpoint_matches=True)

if __name__=='__main__':print(check())


### gpu_rect_explore.py

In [ ]:
%%writefile gpu_rect_explore.py
"""T4 FP32 exploration; FP64 refinement and official metric remain mandatory."""
import numpy as np
import cupy as cp
import gpu_rect_optimizer as exact
source=exact.src.replace('double','float')
for old,new in [('fabs(','fabsf('),('sqrt(','sqrtf('),('cos(','cosf('),('sin(','sinf('),('floor(','floorf('),('fmax(','fmaxf('),('fmin(','fminf(')]:source=source.replace(old,new)
source=source.replace('mp<2e-11','mp<6e-6').replace('sy>1e-24','sy>1e-14').replace('yy>1e-30','yy>1e-20')
module=cp.RawModule(code=source,options=('--std=c++17',),name_expressions=['eval','lbfgs'])
def setup(q,boxes):
    q=cp.asarray(q,dtype=cp.float32);B,n,_=q.shape;boxes=np.asarray(boxes,np.float32).reshape(B,2);nc=int(np.ceil((boxes.max()+6)/1.5))+1
    return q,B,n,nc,cp.asarray(boxes),cp.zeros(B,cp.float32),cp.empty((B,nc*nc),cp.int32),cp.empty((B,n),cp.int32),cp.empty((B,n,2),cp.float32)
def energy(q,boxes):
    q,B,n,nc,bb,t,h,l,c=setup(q,boxes);out=cp.empty((B,2),cp.float32);g=cp.empty_like(q)
    module.get_function('eval')((B,),(256,),(q,g,out,np.int32(n),np.int32(nc),bb,t,h,l,c));return cp.asnumpy(out),cp.asnumpy(g)
def quench(q,boxes,steps=350):
    q,B,n,nc,bb,t,h,l,c=setup(q,boxes);M=8;work=cp.empty((B,5,n,3),cp.float32);hist=cp.empty((B,2,M,n,3),cp.float32);out=cp.empty((B,3),cp.float32)
    module.get_function('lbfgs')((B,),(256,),(q,work,hist,out,np.int32(n),np.int32(nc),bb,t,h,l,c,np.int32(steps),np.int32(M)));return q,out
def check():
    rng=np.random.default_rng(224);q=rng.uniform(.4,2.6,(2,9,3));boxes=np.array([[3.1,3.2],[3.2,3.1]])
    e,g=energy(q,boxes);ee,gg=exact.energy(q,boxes)
    assert np.max(abs(e-ee))<3e-4 and np.max(abs(g-gg))<3e-4
    return dict(fp32_energy_error=float(np.max(abs(e-ee))),fp32_gradient_error=float(np.max(abs(g-gg))),fp64=exact.check())


### gpu_upright_rows.py

In [ ]:
%%writefile gpu_upright_rows.py
"""Batched CUDA upright refill bounds using exact weighted whole-row DP.

Each row mixes left/right wall lattice phases with exact 1-D greedy MIS.
Whole rows use a weighted path MIS; a transposed pass handles columns.
This is a feasible seed family, not global 2-D MIS. Soft core overlaps are
allowed only during exploration and must subsequently be repaired/certified.
"""
import cupy as cp
import numpy as np
CUDA=r'''
extern "C" __global__ void refill(const float*core,int m,int w,int h,float f,
 int total,float soft,float*out,int*capacity){
 int b=blockIdx.x,swap=blockIdx.y,tid=threadIdx.x;
 int width=swap?h:w,height=swap?w:h,rows=2*height,job=2*b+swap;
 __shared__ float tx[192],ty[192],co[192],si[192],positions[64][32];
 __shared__ int count[64],dp[65],choice[64];
 for(int k=tid;k<m;k+=blockDim.x){float a=core[(b*m+k)*3+2];
  tx[k]=core[(b*m+k)*3+(swap?1:0)];ty[k]=core[(b*m+k)*3+(swap?0:1)];
  co[k]=swap?sinf(a):cosf(a);si[k]=swap?cosf(a):sinf(a);
 }__syncthreads();
 if(tid<rows){
  float y=.5f+tid/2+(tid%2)*f,last=-10.f;int n=0;
  for(int j=0;j<2*width;j++){
   float x=.5f+j/2+(j%2)*f;if(x-last<.999998f)continue;
   bool good=true;
   for(int k=0;k<m;k++){
    float dx=x-tx[k],dy=y-ty[k],c=co[k],s=si[k];
    float rad=.5f*(1.f+fabsf(c)+fabsf(s));
    float z=fmaxf(fmaxf(fabsf(dx),fabsf(dy)),fmaxf(fabsf(dx*c+dy*s),fabsf(-dx*s+dy*c)));
    if(z<rad-soft-2e-6f){good=false;break;}
   }
   if(good){positions[tid][n++]=x;last=x;}
  }count[tid]=n;
 }__syncthreads();
 if(tid==0){
  dp[0]=0;
  for(int r=0;r<rows;r++){int take=count[r]+(r>=1?dp[r-1]:0);choice[r]=take>dp[r];dp[r+1]=max(dp[r],take);}
  capacity[job]=m+dp[rows];
  float*q=out+job*total*3;
  for(int k=0;k<m;k++)for(int d=0;d<3;d++)q[3*k+d]=core[(b*m+k)*3+d];
  int n=m;
  for(int r=rows-1;r>=0&&n<total;){
   if(!choice[r]){r--;continue;}
   float y=.5f+r/2+(r%2)*f;
   for(int j=0;j<count[r]&&n<total;j++){
    float x=positions[r][j];q[3*n]=swap?y:x;q[3*n+1]=swap?x:y;q[3*n+2]=0.f;n++;
   }r-=2;
  }
 }
}'''
kernel=cp.RawKernel(CUDA,'refill')
def refill_batch(core,w,h,f,total,soft=.008):
    core=cp.asarray(core,dtype=cp.float32);B,m,_=core.shape
    assert m<=192 and w<=31 and h<=31 and 0<=f<1
    out=cp.zeros((2*B,total,3),cp.float32);counts=cp.empty(2*B,cp.int32)
    kernel((B,2),(128,),(core,np.int32(m),np.int32(w),np.int32(h),np.float32(f),np.int32(total),np.float32(soft),out,counts))
    return out,counts
def check():
    from upright_refill import gaps
    p,c=refill_batch(np.empty((1,0,3)),3,3,.2,9,soft=0.)
    counts=cp.asnumpy(c);q=cp.asnumpy(p);assert min(counts)==9 and all(gaps(r)>-2e-6 for r in q)
    core=np.array([[[1.5,1.5,.3]]]);p,c=refill_batch(core,4,4,.1,8,soft=0.)
    counts=cp.asnumpy(c);q=cp.asnumpy(p)
    good=[i for i,k in enumerate(counts)if k>=8];assert good and all(gaps(q[i])>-3e-6 for i in good)
    return dict(empty_grid_capacity=9,tilted_feasible_row_families=len(good))


### upright_refill.py

In [ ]:
%%writefile upright_refill.py
"""Independent finite-candidate upright refill inspired by Chris Deotte FTMC.

Simplicial reductions preserve the maximum independent set. HiGHS MILP is
exact only when it proves optimality; time-limited feasible seeds are labelled.
This module never certifies a continuous packing or changes unit-square sizes.
"""
import time, math
import numpy as np
from scipy.spatial import cKDTree
from scipy.optimize import milp, Bounds, LinearConstraint
from scipy.sparse import coo_matrix

def half(a):return .5*(abs(np.cos(a))+abs(np.sin(a)))

def gaps(p):
    if len(p)<2:return 1.
    pairs=cKDTree(p[:,:2]).query_pairs(1.415,output_type='ndarray')
    if not len(pairs):return 1.
    i,j=pairs.T;d=p[j,:2]-p[i,:2];a,b=p[i,2],p[j,2]
    proj=np.maximum.reduce([abs(d[:,0]*np.cos(t)+d[:,1]*np.sin(t))for t in [a,a+np.pi/2,b,b+np.pi/2]])
    return float(np.min(proj-.5*(1+abs(np.cos(a-b))+abs(np.sin(a-b)))))

def core_gaps(xy,core):
    if not len(core):return np.full(len(xy),100.)
    d=xy[:,None,:]-core[None,:,:2];a=core[:,2];c,s=np.cos(a),np.sin(a)
    rad=.5*(1+abs(c)+abs(s))
    sep=np.maximum.reduce([abs(d[:,:,0]),abs(d[:,:,1]),abs(d[:,:,0]*c+d[:,:,1]*s),abs(-d[:,:,0]*s+d[:,:,1]*c)])-rad
    # On upright axes the support is also (1+|cos a|+|sin a|)/2.
    return sep.min(1)

def candidates(W,H,core,extra=None,soft=0.,depth=4):
    xs=np.unique(np.r_[np.arange(.5,W-.499999,1.),W-np.arange(.5,W-.499999,1.)])
    ys=np.unique(np.r_[np.arange(.5,H-.499999,1.),H-np.arange(.5,H-.499999,1.)])
    X,Y=np.meshgrid(xs,ys);points=[np.column_stack((X.ravel(),Y.ravel()))]
    if extra is not None and len(extra):points.append(np.asarray(extra)[:,:2])
    margin=3e-7-soft
    # Intersect the four SAT strips: exact Minkowski-obstacle sections.
    for t in core:
        c,s=np.cos(t[2]),np.sin(t[2]);r=.5*(1+abs(c)+abs(s));extent=r
        for swap,grid,lim in [(False,ys,W),(True,xs,H)]:
            fixed=grid-t[0 if swap else 1]
            axes=[(1.,0.),(c,s),(-s,c)] if not swap else [(1.,0.),(s,c),(c,-s)]
            lo=np.full(len(grid),-extent);hi=np.full(len(grid),extent);valid=abs(fixed)<extent+1e-8
            for a,b in axes[1:]:
                if abs(a)<1e-10:valid&=abs(b*fixed)<=r
                else:
                    l=(-r-b*fixed)/a;h=(r-b*fixed)/a;lo=np.maximum(lo,np.minimum(l,h));hi=np.minimum(hi,np.maximum(l,h))
            valid&=lo<=hi
            center=t[1 if swap else 0]
            for bound,sign in [(lo,-1),(hi,1)]:
                for k in range(depth+1):
                    moving=center+bound+sign*(k+margin)
                    z=np.column_stack((grid[valid],moving[valid]))if swap else np.column_stack((moving[valid],grid[valid]))
                    points.append(z)
    xy=np.vstack(points);xy=xy[(xy[:,0]>=.5-1e-8)&(xy[:,0]<=W-.5+1e-8)&(xy[:,1]>=.5-1e-8)&(xy[:,1]<=H-.5+1e-8)]
    if not len(xy):return xy
    _,ids=np.unique(np.round(xy,7),axis=0,return_index=True);xy=xy[np.sort(ids)]
    keep=[]
    for start in range(0,len(xy),512):keep.extend(np.flatnonzero(core_gaps(xy[start:start+512],core)>=-soft-2e-8)+start)
    return xy[np.asarray(keep,dtype=int)]

def independent_set(xy,target=None,seconds=.35):
    started=time.perf_counter();n=len(xy)
    if not n:return [],dict(candidates=0,simplicial=0,milp=0,optimal=True)
    pairs=cKDTree(xy).query_pairs(math.sqrt(2),output_type='ndarray')
    pairs=pairs[(abs(xy[pairs[:,0]]-xy[pairs[:,1]])<1.-1e-8).all(1)] if len(pairs)else pairs
    adj=[set()for _ in range(n)]
    for a,b in pairs:adj[int(a)].add(int(b));adj[int(b)].add(int(a))
    live=set(range(n));fixed=[];changed=True
    while changed:
        changed=False
        for v in sorted(live,key=lambda i:len(adj[i]&live)):
            if v not in live:continue
            nb=adj[v]&live
            if len(nb)<=1 or (len(nb)<=12 and all((nb-{u})<=adj[u]for u in nb)):
                fixed.append(v);live-=nb|{v};changed=True
        if target is not None and len(fixed)>=target:return fixed[:target],dict(candidates=n,simplicial=len(fixed),milp=0,optimal=False,reason='target reached')
    def greedy(vertices):
        available=set(vertices);result=[]
        while available:
            u=min(available,key=lambda k:len(adj[k]&available));result.append(u);available-=adj[u]|{u}
        return result
    selected=fixed.copy();visited=set();components=[]
    for v in live:
        if v in visited:continue
        stack=[v];visited.add(v);comp=[]
        while stack:
            u=stack.pop();comp.append(u)
            for z in (adj[u]&live)-visited:visited.add(z);stack.append(z)
        components.append(comp)
    guesses=[greedy(comp)for comp in components]
    if target is not None and len(fixed)+sum(map(len,guesses))>=target:
        return (fixed+sum(guesses,[]))[:target],dict(candidates=n,simplicial=len(fixed),milp=0,optimal=False,reason='feasible greedy target')
    solved=0;optimal=True
    for comp,guess in zip(components,guesses):
        remaining=seconds-(time.perf_counter()-started)
        if remaining<.02 or len(comp)>1800:selected+=guess;optimal=False;continue
        index={v:i for i,v in enumerate(comp)};edges=[(index[u],index[v])for u in comp for v in adj[u]&set(comp)if u<v]
        row=np.repeat(np.arange(len(edges)),2);col=np.asarray(edges,dtype=int).ravel()
        A=coo_matrix((np.ones(len(col)),(row,col)),shape=(len(edges),len(comp))).tocsc()
        fit=milp(-np.ones(len(comp)),integrality=np.ones(len(comp)),bounds=Bounds(0,1),constraints=LinearConstraint(A,-np.inf,1),options={'time_limit':max(.02,remaining),'mip_rel_gap':0.,'threads':1})
        take=guess
        if fit.x is not None:
            trial=[comp[i]for i in np.flatnonzero(fit.x>.5)]
            if len(trial)>len(take)and all(v not in adj[u]for i,u in enumerate(trial)for v in trial[i+1:]):take=trial
        selected+=take;solved+=1;optimal&=fit.status==0
    return selected,dict(candidates=n,simplicial=len(fixed),milp=solved,optimal=bool(optimal),seconds=time.perf_counter()-started)

def refill(W,H,core,total,extra=None,seconds=.35,soft=0.):
    need=total-len(core)
    if need<0:return None,dict(reason='too many tilted squares')
    xy=candidates(W,H,core,extra,soft);ids,report=independent_set(xy,need,seconds)
    report.update(needed=need,found=len(ids),soft=soft)
    if len(ids)<need:return None,report
    upright=np.column_stack((xy[ids[:need]],np.zeros(need)))
    return np.vstack((core,upright)),report

def check():
    xy=np.array([[.5,.5],[1.5,.5],[.5,1.5],[1.5,1.5],[1.,1.]])
    ids,r=independent_set(xy,seconds=2.)
    assert len(ids)==4,(ids,r)
    p,r=refill(3.,3.,np.empty((0,3)),9,seconds=2.)
    assert p is not None and len(p)==9 and gaps(p)>-1e-10
    t=np.array([[1.5,1.5,.3]]);p,r=refill(4.,4.,t,8,seconds=2.)
    assert p is not None and gaps(p)>-1e-7,(p,r)
    return dict(conflict_graph_known_optimum=4,empty_3x3_capacity=9,tilted_seed_count=len(p),tilted_seed_gap=gaps(p))


### gpu_ftmc_refill.py

In [ ]:
%%writefile gpu_ftmc_refill.py
"""Independent FTMC/refill campaign, following Deotte's public architecture.

Novelty archive, rational-slope core families, FP32 T4 screening, FP64 final
quench, exact finite-candidate upright refill and unchanged official admission.
All expensive execution belongs to Kaggle. No local search is required.
"""
import os
for k in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS']:os.environ[k]='1'
import argparse,json,math,time,hashlib,warnings
from pathlib import Path
import numpy as np
import cupy as cp
from scipy.spatial import cKDTree
from scipy.optimize import linprog
from scipy.sparse import coo_matrix
from upright_refill import half,gaps,refill,check as refill_check
from gpu_rect_explore import quench as explore,check as gpu_check
from gpu_rect_optimizer import quench as precise
from gpu_upright_rows import refill_batch,check as row_check
from solver import normalize
warnings.filterwarnings('ignore',message='Unrecognized options detected')

def atom_json(path,value):
    p=Path(path);t=p.with_suffix(p.suffix+'.tmp');t.write_text(json.dumps(value,indent=2),encoding='utf8');t.replace(p)
def atom_npz(path,bank):
    p=Path(path);t=p.with_suffix(p.suffix+'.tmp')
    with t.open('wb')as f:np.savez_compressed(f,**bank)
    t.replace(p)
def dimensions(p):
    e=half(p[:,2]);return (p[:,:2]+e[:,None]).max(0)-(p[:,:2]-e[:,None]).min(0)
def excess(p,w,h):return float(max(dimensions(p)-[w,h]))
def repair(p,clearance=3e-9):
    p=normalize(p,0);pairs=cKDTree(p[:,:2]).query_pairs(1.415,output_type='ndarray')
    if len(pairs):
        i,j=pairs.T;d=p[j,:2]-p[i,:2];a,b=p[i,2],p[j,2]
        z=np.maximum.reduce([abs(d[:,0]*np.cos(t)+d[:,1]*np.sin(t))for t in [a,a+np.pi/2,b,b+np.pi/2]])
        r=.5*(1+abs(np.cos(a-b))+abs(np.sin(a-b)));ratio=max(1.,float(np.max((r+clearance)/np.maximum(z,1e-30))))
        if ratio>1.002:return None
        p[:,:2]*=ratio
    return normalize(p,0)

def rect_lp(p,w,h,rounds=3):
    p=normalize(p,0);n=len(p);f=excess(p,w,h)
    for _ in range(rounds):
        a=p[:,2];c,s=np.cos(a),np.sin(a);e=half(a);pairs=cKDTree(p[:,:2]).query_pairs(1.7,output_type='ndarray')
        if not len(pairs):return repair(p)
        i,j=pairs.T;d=p[j,:2]-p[i,:2];ux=np.column_stack((c[i],-s[i],c[j],-s[j]));uy=np.column_stack((s[i],c[i],s[j],c[j]))
        pr=d[:,0,None]*ux+d[:,1,None]*uy;k=abs(pr).argmax(1);ii=np.arange(len(i));sign=np.where(pr[ii,k]>=0,1.,-1.);vx=ux[ii,k]*sign;vy=uy[ii,k]*sign
        cols=[np.column_stack((2*i,2*i+1,2*j,2*j+1))];vals=[np.column_stack((vx,vy,-vx,-vy))];rhs=[-.5*(1+abs(np.cos(a[i]-a[j]))+abs(np.sin(a[i]-a[j])))-4e-9];rows=[np.repeat(ii,4)];offset=len(i);ids=np.arange(n)
        for dim,span in [(0,w),(1,h)]:
            for upper in [False,True]:
                cols.append(np.column_stack((2*ids+dim,np.full(n,2*n))));vals.append(np.tile([1. if upper else -1.,-1. if upper else 0.],(n,1)))
                rhs.append((span if upper else 0)-e-4e-9);rows.append(np.repeat(np.arange(offset,offset+n),2));offset+=n
        A=coo_matrix((np.concatenate([v.ravel()for v in vals]),(np.concatenate(rows),np.concatenate([v.ravel()for v in cols]))),shape=(offset,2*n+1)).tocsr()
        obj=np.zeros(2*n+1);obj[-1]=1.;fit=linprog(obj,A_ub=A,b_ub=np.concatenate(rhs),bounds=[(x-.065,x+.065)for x in p[:,:2].ravel()]+[(max(0.,f-.08),f+.03)],method='highs',options={'time_limit':.7,'primal_feasibility_tolerance':1e-9,'dual_feasibility_tolerance':1e-9,'threads':1})
        if not fit.success:break
        q=p.copy();q[:,:2]=fit.x[:2*n].reshape(n,2);q=repair(q)
        if q is None or excess(q,w,h)>=f-1e-9:break
        p=q;f=excess(p,w,h)
    return repair(p)

def certify(p):
    import sys,csv,io
    sys.path.insert(0,str(Path('research').resolve()));import official_metric as metric
    n=len(p);rows=[dict(n=str(n),square_id=str(i),x=format(x,'.17g'),y=format(y,'.17g'),angle=format(a,'.17g'))for i,(x,y,a)in enumerate(p)]
    raw=metric._evaluate_puzzle_worker((n,rows))
    if raw[3]!='ok':return False
    import pandas as pd
    s=io.StringIO();pd.read_csv(io.StringIO(pd.DataFrame(rows).to_csv(index=False))).to_csv(s,index=False,float_format='%.17g');s.seek(0)
    rt=metric._evaluate_puzzle_worker((n,list(csv.DictReader(s))))
    return rt[3]=='ok' and abs(raw[1]-rt[1])<1e-10

def fingerprint(p):
    a=(p[:,2]+np.pi/4)%(np.pi/2)-np.pi/4;tilted=p[abs(a)>.018]
    hist=np.histogram(a,bins=np.linspace(-np.pi/4,np.pi/4,13))[0]
    if len(tilted):
        xy=tilted[:,:2]-tilted[:,:2].min(0);span=np.maximum(np.ptp(xy,axis=0),1.)
        shape=np.histogram2d(xy[:,0]/span[0],xy[:,1]/span[1],bins=(5,8),range=((0,1.000001),(0,1.000001)))[0].ravel()
    else:shape=np.zeros(40)
    signature=np.r_[len(tilted),hist,shape]
    return hashlib.sha256(signature.astype(np.int32).tobytes()).hexdigest()[:12]

def cold_core(w,h,f,m,rng,angle):
    # A genuinely new rational-slope lattice phase; no incumbent coordinates.
    c,s=np.cos(angle),np.sin(angle);extent=.5*(abs(c)+abs(s));W,H=w+f,h+f
    uv=np.array([(x,y)for x in range(-36,37)for y in range(-36,37)],float)
    uv+=rng.uniform(-.5,.5,2);xy=uv@np.array([[c,s],[-s,c]])+[W/2,H/2]
    mask=(xy[:,0]>=extent)&(xy[:,0]<=W-extent)&(xy[:,1]>=extent)&(xy[:,1]<=H-extent);xy=xy[mask]
    if len(xy)<m:return None
    # Prefer a long diagonal band; vary its offset and taper from scratch.
    axis=np.array([w,h],float);axis/=np.linalg.norm(axis);normal=np.array([-axis[1],axis[0]])
    score=abs((xy-[W/2,H/2])@normal-rng.uniform(-.4,.4))+.015*abs((xy-[W/2,H/2])@axis)
    return np.column_stack((xy[np.argsort(score)[:m]],np.full(m,angle)))

def mutations(p,w,h,d,rng,B,mode,archive):
    a=(p[:,2]+np.pi/4)%(np.pi/2)-np.pi/4;tilt=abs(a)>.018;core=p[tilt].copy();upright=p[~tilt]
    if len(core)<4 or len(core)>160:return [],[],[]
    m0=len(core);offset=1 if mode==1 else -int(rng.integers(1,4))if mode==2 else 0;m=min(len(p)-2,max(4,m0+offset));f0=excess(p,w,h)
    seeds=[];fs=[];meta=[];angles=np.radians([26.565051,28.072487,33.690068,36.869898,40.,45.,-26.565051,-33.690068])
    for b in range(B):
        q=core.copy();angle=float(angles[(b+int(rng.integers(len(angles))))%len(angles)])
        if mode==5:
            q=cold_core(w,h,f0,m,rng,angle)
            if q is None:continue
        elif mode==4 and archive:
            q=archive[int(rng.integers(len(archive)))].copy();aa=(q[:,2]+np.pi/4)%(np.pi/2)-np.pi/4;q=q[abs(aa)>.018]
            if len(q)<4:continue
        if len(q)>m:
            axis=np.array([w,h],float);score=q[:,:2]@axis;ids=np.argsort(score);drop=ids[:len(q)-m]if b%2 else ids[-(len(q)-m):];q=np.delete(q,drop,axis=0)
        while len(q)<m:
            axis=np.array([w,h],float);axis/=np.linalg.norm(axis);origin=q[rng.integers(len(q))].copy();projection=q[:,:2]@axis;cut=np.median(projection)
            q[projection>cut,:2]+=.42*axis;q[projection<=cut,:2]-=.42*axis;origin[:2]=q[:,:2].mean(0)+rng.normal(0,.15,2)
            q=np.vstack((q,origin))
        if mode==3 and len(upright):
            tree=cKDTree(q[:,:2]);near=tree.query(upright[:,:2])[0];ids=np.argsort(near)[:min(10,len(upright))]
            u=upright[int(rng.choice(ids))].copy();u[2]=float(np.median(q[:,2]));tip=np.argmax(np.linalg.norm(q[:,:2]-u[:2],axis=1));q[tip]=u
        if mode!=5:
            centre=q[:,:2].mean(0);old=.25*np.angle(np.mean(np.exp(4j*q[:,2])))
            delta=((angle-old+np.pi/4)%(np.pi/2)-np.pi/4) if b%3==0 else rng.normal(0,.018 if b%2 else .065)
            c,s=np.cos(delta),np.sin(delta);q[:,:2]=(q[:,:2]-centre)@np.array([[c,s],[-s,c]])+centre;q[:,2]+=delta
            q[:,0]+=(q[:,1]-centre[1])*rng.normal(0,.015);q[:,:2]+=rng.normal(0,.004,q[:,:2].shape)
        q=normalize(q,0);wh=dimensions(q);box=np.array([w+f0,h+f0]);q[:,:2]+=(box-wh)/2
        depth=[.002,.008,.02,.045,.08,.12][b%6];ftry=max(.12,f0-depth)
        seeds.append(q);fs.append(ftry);meta.append(dict(mode=mode,core_count=m,angle=angle,cold=mode==5))
    return seeds,fs,meta

EXTRACT=r'''
extern "C" __global__ void extract(const double*sorted,int n,int donor,unsigned long long*best){
 int z=blockIdx.x*blockDim.x+threadIdx.x;if(z>=31*31*63)return;
 int d=z%63+1,t=z/63,w=t/31+1,h=t%31+1,m=w*h+d;if(m>n)return;
 double f=sorted[t*n+m-1]+3e-8;if(f<=3e-8||f>=.999999)return;
 unsigned long long q=(unsigned long long)ceil(f*1e9),job=donor*(31*31*63)+z;atomicMin(best+(w*32+h)*64+d,(q<<32)|job);
}'''

def build_library(baseline,examples,limit=320):
    from gpu_fault_composition import read_csv,crop_data
    start=time.perf_counter();bank={};variants={};provenance={};donors=[]
    z=np.load(examples);meta=z['meta_39']
    for k,row in enumerate(meta):
        w,h,d=map(int,row[:3])
        for stage in ['before','after']:
            p=z[f'{stage}_{k}'];assert len(p)==w*h+d
            for swap in [False,True]:
                q=p.copy();ww,hh=w,h
                if swap:q[:,:2]=q[:,[1,0]];q[:,2]=np.pi/2-q[:,2];ww,hh=h,w
                key=f'{ww}_{hh}_{d}';q=normalize(q,0)
                variants[key+f'__{stage}{k}']=q
                if key not in bank or excess(q,ww,hh)<excess(bank[key],ww,hh):bank[key]=q;provenance[key]=f'Chris Deotte published {stage}_{k}, dataset v2'
    atom_npz('artifacts/night_seed_gadgets.npz',bank);atom_npz('artifacts/night_seed_variants.npz',variants);atom_json('research/night_priorities.json',{})
    full=read_csv(baseline)
    for n,p in full.items():
        if n<5:continue
        for swap in [False,True]:
            q=p.copy()
            if swap:q[:,:2]=q[:,[1,0]];q[:,2]=np.pi/2-q[:,2]
            for sx,sy in [(1,1),(-1,1),(1,-1),(-1,-1)]:
                t=q.copy();t[:,0]*=sx;t[:,1]*=sy;t[:,2]*=sx*sy;donors.append((normalize(t,0),n))
    kernel=cp.RawKernel(EXTRACT,'extract');best=cp.full(32*32*64,np.uint64(0xffffffffffffffff));
    for i,(p,n)in enumerate(donors):
        r,s=crop_data(p);kernel(((31*31*63+127)//128,),(128,),(s,np.int32(len(p)),np.int32(i),best))
        if i%500==0:cp.cuda.Stream.null.synchronize();print(json.dumps(dict(stage='GPU crop library',done=i,total=len(donors),seconds=time.perf_counter()-start)),flush=True)
    kk=cp.asnumpy(best).reshape(32,32,64);rank=[]
    incumbent={n:excess(p,math.isqrt(n),math.isqrt(n))for n,p in full.items()}
    for w,h,d in np.argwhere(kk<np.uint64(0xffffffffffffffff)):
        f=(int(kk[w,h,d])>>32)*1e-9
        if w+h>54 or w*h+d>600 or min(w,h)<2:continue
        leverage=sum(max(0.,incumbent.get(k*k+d,1.)-(f-.04))/math.sqrt(k*k+d)for k in range(max(int(w),int(h)),32)if k*k+d<=1000)
        rank.append((leverage,int(w),int(h),int(d),f))
    rank.sort(reverse=True)
    for score,w,h,d,f in rank[:limit]:
        value=int(kk[w,h,d]);di=(value&0xffffffff)//(31*31*63);p,n=donors[di];e=half(p[:,2]);r=np.maximum.reduce((p[:,0]+e-w,p[:,1]+e-h,np.zeros(len(p))));q=normalize(p[np.argsort(r,kind='stable')[:w*h+d]],0);key=f'{w}_{h}_{d}'
        if key not in bank or excess(q,w,h)<excess(bank[key],w,h):bank[key]=q;provenance[key]=f'Certified union crop n={n}, D4 donor {di}'
    atom_npz('artifacts/night_seed_gadgets.npz',bank);atom_json('research/night_seed_provenance.json',provenance)
    priority={f'{w}_{h}_{d}':score for score,w,h,d,f in rank[:limit]}
    atom_json('research/night_priorities.json',priority);print(json.dumps(dict(stage='GPU library complete',gadgets=len(bank),seconds=time.perf_counter()-start)),flush=True)

def worker(device,seconds=21600,replicas=64,seed=0):
    cp.cuda.Device(device).use();start=time.perf_counter();rng=np.random.default_rng(20261024+seed+device*19937)
    stats=dict(device=device,seed=seed,batches=0,core_trials=0,refill_success=0,full_quenches=0,accepted=0,cuda_seconds=0.,families={},official_rejected=0)
    checks=dict(refill=refill_check(),gpu=gpu_check(),cuda_rows=row_check());atom_json(f'research/worker{device}_checks.json',checks);print(json.dumps(dict(stage='preflight passed',device=device,checks=checks)),flush=True)
    initial=np.load('artifacts/night_seed_gadgets.npz');bank={k:initial[k]for k in initial.files};archive={k:[p.copy()]for k,p in bank.items()};wins={};records=[]
    if Path('artifacts/night_seed_variants.npz').exists():
        with np.load('artifacts/night_seed_variants.npz')as variants:
            for name in variants.files:
                root=name.split('__')[0]
                if root in archive:archive[root].append(variants[name])
    existing=Path(f'artifacts/worker{device}_gadgets.npz')
    if existing.exists():
        with np.load(existing)as old:
            for key in old.files:
                root=key.split('__')[0];w,h,d=map(int,root.split('_'));p=old[key];archive.setdefault(root,[]).append(p)
                if root not in bank or excess(p,w,h)<excess(bank[root],w,h):bank[root]=p
                wins[key]=p
    priority=json.loads(Path('research/night_priorities.json').read_text());keys=list(bank);visits={k:0 for k in keys}
    event1,event2=cp.cuda.Event(),cp.cuda.Event()
    def checkpoint():
        atom_npz(existing,wins);stats.update(seconds=time.perf_counter()-start,gadgets=len(wins),latest=records[-1]if records else None,records=records[-300:]);atom_json(f'research/worker{device}_status.json',stats)
    checkpoint()
    while time.perf_counter()-start<seconds:
        weights=np.array([(.08+100*priority.get(k,0.)) / math.sqrt(1+visits[k]) for k in keys]);weights/=weights.sum();key=keys[int(rng.choice(len(keys),p=weights))];visits[key]+=1
        w,h,d=map(int,key.split('_'));p=bank[key];f0=excess(p,w,h);mode=(stats['batches']+device)%6;stats['batches']+=1;stats['families'][str(mode)]=stats['families'].get(str(mode),0)+1
        source=archive[key][int(rng.integers(len(archive[key])))];seeds,fs,recipes=mutations(source,w,h,d,rng,replicas,mode,archive[key])
        if not seeds:continue
        stats['core_trials']+=len(seeds);fs=np.minimum(np.array(fs),f0-.0005);loose=min(.995,f0+.035);event1.record()
        cq,out=explore(np.asarray(seeds),np.tile([w+loose,h+loose],(len(seeds),1)),steps=500)
        event2.record();event2.synchronize();stats['cuda_seconds']+=cp.cuda.get_elapsed_time(event1,event2)/1000
        scores=cp.asnumpy(out);cores=cp.asnumpy(cq).astype(float);event1.record()
        packed,capacity=refill_batch(cq,w,h,loose,w*h+d,soft=.008)
        event2.record();event2.synchronize();stats['cuda_seconds']+=cp.cuda.get_elapsed_time(event1,event2)/1000
        row_counts=cp.asnumpy(capacity);row_seeds=cp.asnumpy(packed).astype(float);filled=[];filled_fs=[];which=[];seen=set();fill_reports=[]
        for job in np.argsort(-row_counts,kind='stable'):
            b=int(job)//2
            if row_counts[job]<w*h+d:break
            if not np.isfinite(scores[b]).all()or not np.isfinite(cores[b]).all()or scores[b,1]>.01:continue
            sig=fingerprint(cores[b])+str(int(job)%2)
            if sig in seen:continue
            seen.add(sig);filled.append(row_seeds[job]);filled_fs.append(fs[b]);which.append(b);stats['refill_success']+=1
            if len(filled)>=8:break
        # Only a few near-capacity failures reach the more expensive 2-D MILP.
        near=np.argsort(-row_counts.reshape(-1,2).max(1),kind='stable');attempts=0
        import signal
        previous=signal.getsignal(signal.SIGALRM)
        def timed_out(signum,frame):raise TimeoutError('Refill seed budget')
        signal.signal(signal.SIGALRM,timed_out)
        try:
            for b in near:
                if len(filled)>=12 or attempts>=4:break
                if row_counts.reshape(-1,2)[b].max()<w*h+d-4:continue
                if not np.isfinite(cores[b]).all()or scores[b,1]>.01:continue
                attempts+=1;extra=p[abs((p[:,2]+np.pi/4)%(np.pi/2)-np.pi/4)<.018]
                try:
                    signal.setitimer(signal.ITIMER_REAL,1.25)
                    q,fr=refill(w+loose,h+loose,cores[b],w*h+d,extra,seconds=.25,soft=.008)
                except TimeoutError:q=None;fr=dict(timeout=True)
                finally:signal.setitimer(signal.ITIMER_REAL,0)
                fill_reports.append(fr)
                if q is not None:filled.append(q);filled_fs.append(fs[b]);which.append(int(b));stats['refill_success']+=1
        finally:signal.signal(signal.SIGALRM,previous)
        gained=False;best=f0
        if filled:
            R=16;qbatch=np.repeat(np.asarray(filled),R,axis=0);ff=np.repeat(np.array(filled_fs),R);origins=np.repeat(np.array(which),R)
            for j in range(len(qbatch)):
                if j%R==0:continue
                mask=abs((qbatch[j,:,2]+np.pi/4)%(np.pi/2)-np.pi/4)>.018
                qbatch[j,mask,:2]+=rng.normal(0,.0003,(int(mask.sum()),2));qbatch[j,mask,2]+=rng.normal(0,.0004,int(mask.sum()))
            event1.record();cq,out=explore(qbatch,np.column_stack((w+ff,h+ff)),steps=1800)
            event2.record();event2.synchronize();stats['cuda_seconds']+=cp.cuda.get_elapsed_time(event1,event2)/1000;rough=cp.asnumpy(out);q32=cp.asnumpy(cq);stats['full_quenches']+=len(qbatch)
            pick=np.argsort(rough[:,1])[:8];event1.record()
            cq,out=precise(q32[pick],np.column_stack((w+ff[pick]+2e-5,h+ff[pick]+2e-5)),steps=1100)
            event2.record();event2.synchronize();stats['cuda_seconds']+=cp.cuda.get_elapsed_time(event1,event2)/1000
            q64=cp.asnumpy(cq);end=cp.asnumpy(out)
            for k,b in enumerate(pick):
                if not np.isfinite(end[k]).all()or not np.isfinite(q64[k]).all()or end[k,1]>3e-5:continue
                q=rect_lp(q64[k],w,h);f=excess(q,w,h)if q is not None else 99.
                if q is None or f>f0+.025 or gaps(q)<1e-10:continue
                signature=fingerprint(q);known={fingerprint(t)for t in archive[key]}
                improved=f<f0-2e-7
                if not improved and signature in known:continue
                if not certify(q):stats['official_rejected']+=1;continue
                if signature not in known:archive[key].append(q);archive[key]=sorted(archive[key],key=lambda t:excess(t,w,h))[:6]
                wins[key+'__'+signature]=q
                if improved:
                    bank[key]=q;f0=f;best=f;gained=True;stats['accepted']+=1
                    records.append(dict(key=key,before=excess(p,w,h),after=f,mode=mode,recipe=recipes[int(origins[int(b)])],signature=signature,minimum_gap=gaps(q),official_raw_and_pandas=True))
                    print(json.dumps(dict(stage='new certified gadget',device=device,**records[-1])),flush=True)
        stats['last_batch']=dict(key=key,mode=mode,core_candidates=len(seeds),refilled=len(filled),fill_reports=fill_reports[-4:],before=excess(p,w,h),after=best)
        checkpoint()
        if stats['batches']%4==0:print(json.dumps({k:v for k,v in stats.items()if k not in ['records','latest','last_batch']}),flush=True)
    checkpoint();print(json.dumps(dict(stage='worker complete',device=device,seconds=time.perf_counter()-start,accepted=stats['accepted'])),flush=True)

if __name__=='__main__':
    ap=argparse.ArgumentParser();ap.add_argument('--library',action='store_true');ap.add_argument('--baseline');ap.add_argument('--examples');ap.add_argument('--device',type=int,default=0);ap.add_argument('--seconds',type=float,default=21600);ap.add_argument('--replicas',type=int,default=64);ap.add_argument('--seed',type=int,default=0);a=ap.parse_args()
    cp.cuda.Device(a.device).use()
    if a.library:build_library(a.baseline,a.examples)
    else:worker(a.device,a.seconds,a.replicas,a.seed)


### gpu_fault_composition.py

In [ ]:
%%writefile gpu_fault_composition.py
"""CUDA crop extraction and minimax staircase composition.

Composition geometry: Chris Deotte's public Multi-Gadget Fault DP notebook.
This implementation performs crop order statistics and the DAG DP on CUDA.
The conservative tip test certifies zero-buffer joins for every f < 1.
Inputs include credited public witnesses and independently generated V9 rows.
"""
import argparse,hashlib,json,math,time
from pathlib import Path
import cupy as cp
import numpy as np
import pandas as pd
from solver import side,normalize,grid
from gpu_families import min_gap
from hybrid_search import export

K=31;D=63;GAP=3e-8
CUDA=r'''
extern "C" __global__ void extract(const double* p,int n,const double* r,
 const double* sorted,int donor,unsigned long long* best){
 int z=blockDim.x*blockIdx.x+threadIdx.x;
 if(z>=31*31*63)return;
 int dm=z%63+1,t=z/63,w=t/31+1,h=t%31+1,m=w*h+dm;
 if(m>n)return;
 double f=sorted[t*n+m-1]+3e-8;
 if(f<=3e-8||f>=.999999)return;
 int br=1,tl=1,count=0;
 for(int i=0;i<n;i++)if(r[t*n+i]<=f-2e-8){
   count++; double x=p[3*i],y=p[3*i+1],a=p[3*i+2];
   double e=.5*(fabs(cos(a))+fabs(sin(a)));
   if(x+e>w+1e-9&&y-e<1.00000001)br=0;
   if(x-e<1.00000001&&y+e>h+1e-9)tl=0;
 }
 if(count<m)return;
 unsigned long long q=(unsigned long long)ceil(f*1e9);
 unsigned int job=donor*(31*31*63)+z;
 unsigned long long key=(q<<32)|job;
 atomicMin(best+(((w*32+h)*64+dm)*4+2*br+tl),key);
}
extern "C" __global__ void dag(int sum,const int* g,const double* f,int ng,
 double* dp,int* pg,int* ps,int* pm){
 int z=blockIdx.x*blockDim.x+threadIdx.x;
 if(z>=31*63*2)return;
 int b=z%2,d=(z/2)%63+1,w=z/(2*63)+1,h=sum-w;
 if(h<1||h>31)return;
 double value=2.;int prev=-1,which=-1,mode=-1;
 for(int i=0;i<ng;i++){
  int gw=g[5*i],gh=g[5*i+1],gd=g[5*i+2],br=g[5*i+3],tl=g[5*i+4];
  if(br!=b||gw>w||gh>h||gd>d||f[i]>=value)continue;
  if(gw==w&&gh==h&&gd==d){value=f[i];which=i;prev=-1;mode=-1;continue;}
  if(gd==d)continue;
  for(int m=0;m<3;m++){
   int x=w-gw-(m==1),y=h-gh-(m==2);
   if(x<1||y<1)continue;
   for(int pb=0;pb<2;pb++){
    if(m==0&&!pb&&!tl)continue;
    int ix=(((x*32+y)*64+(d-gd))*2+pb);
    double cand=fmax(f[i],dp[ix]);
    if(cand<value){value=cand;which=i;prev=ix;mode=m;}
   }
  }
 }
 int ix=(((w*32+h)*64+d)*2+b);
 dp[ix]=value;pg[ix]=which;ps[ix]=prev;pm[ix]=mode;
}
extern "C" __global__ void strips(int height,const double* cost,double* val,int* ph,int* pd){
 int z=blockDim.x*blockIdx.x+threadIdx.x;if(z>=31*64)return;
 int k=z/64+1,d=z%64;if(height>k)return;
 double best=2.;int hh=-1,dd=-1;
 for(int h=1;h<=height;h++)for(int t=0;t<=d;t++){
  double f=t?cost[(k*32+h)*64+t]:0.;
  double p=val[(k*32+height-h)*64+d-t];double v=f+p;
  if(v<best){best=v;hh=h;dd=t;}
 }
 int ix=(k*32+height)*64+d;val[ix]=best;ph[ix]=hh;pd[ix]=dd;
}
'''

# Native f-by-f corner typing is combined with a guaranteed padding direction.
# Type 2 moves upward at global F; type 1 moves rightward. Type 3 retains the
# older unit-corner guarantee, because two padding directions cannot be combined
# merely by assuming that each native corner is independently empty.
CUSHION_CUDA=r'''
extern "C" __global__ void extract_cushion(const double*p,int n,const double*r,
 const double*sorted,int donor,unsigned long long*best){
 int z=blockIdx.x*blockDim.x+threadIdx.x;if(z>=31*31*63)return;
 int d=z%63+1,t=z/63,w=t/31+1,h=t%31+1,m=w*h+d;if(m>n)return;
 double f=sorted[t*n+m-1]+3e-8;if(f<=3e-8||f>=.999999)return;
 int br=1,tl=1,ubr=1,utl=1,count=0;
 for(int i=0;i<n;i++)if(r[t*n+i]<=f-2e-8){
  count++;double x=p[3*i],y=p[3*i+1],a=p[3*i+2],e=.5*(fabs(cos(a))+fabs(sin(a)));
  if(x+e>w-1e-9){if(y-e<f+1e-8)br=0;if(y-e<1.00000001)ubr=0;}
  if(y+e>h-1e-9){if(x-e<f+1e-8)tl=0;if(x-e<1.00000001)utl=0;}
 }
 if(count<m)return;
 unsigned long long q=(unsigned long long)ceil(f*1e9),job=donor*(31*31*63)+z,key=(q<<32)|job;
 unsigned long long*dst=best+(((w*32+h)*64+d)*4);
 atomicMin(dst,key);if(tl)atomicMin(dst+1,key);if(br)atomicMin(dst+2,key);
 if(ubr&&utl)atomicMin(dst+3,key);
}
'''

def read_csv(path):
    df=pd.read_csv(path);return {int(n):g[['x','y','angle']].to_numpy(float) for n,g in df.groupby('n',sort=True)}

def inputs():
    frozen=Path('artifacts/fault_source_donors.npz')
    if frozen.exists():
        base=read_csv('research/latest/kirill.csv');sources={n:'Kirill V6 / credited public portfolio' for n in base}
        for n in range(1,1001):base.setdefault(n,grid(n));sources.setdefault(n,'grid')
        labels=json.loads(Path('artifacts/fault_source_labels.json').read_text())
        with np.load(frozen) as bank:donors=[(bank[k],labels[k]) for k in bank.files]
        return base,sources,donors
    base=read_csv('artifacts/sliding_candidate_submission.csv');sources={n:'V9' for n in base}
    for name in ['ensemble','alex']:
        pth=Path('research/latest')/(name+'.csv')
        if not pth.exists():continue
        for n,p in read_csv(pth).items():
            if n not in base or side(p)<side(base[n])-1e-10:base[n]=p;sources[n]=name
    for n in range(1,1001):base.setdefault(n,grid(n));sources.setdefault(n,'grid')
    donors=[];seen=set()
    def add(p,label):
        p=normalize(p,0);L=side(p)
        if len(p)<=int(math.floor(L+1e-7))**2 or L>=32 or len(p)<5:return
        digest=hashlib.sha256(np.round(p,7).tobytes()).hexdigest()
        if digest not in seen:seen.add(digest);donors.append((p,label))
    for n,p in base.items():add(p,sources[n]+':'+str(n))
    for name in ['sliding_refined','clipped_refined','sliding_relaxed']:
        pth=Path('artifacts')/(name+'_packings.npz')
        with np.load(pth) as b:
            for n in b.files:add(b[n],name+':'+n)
    extra=Path('research/latest/couzo_big.csv')
    if extra.exists():
        # External exact-witness conversions, credited to Couzo/original authors.
        # Large donors are cropped; the large layouts themselves are not submitted.
        df=pd.read_csv(extra);starts=np.flatnonzero(df.square_id.to_numpy()==0);stops=np.r_[starts[1:],len(df)]
        variants=[]
        for first,last in zip(starts,stops):
            part=df.iloc[first:last];n=int(part.n.iloc[0]);assert len(part)==n and np.array_equal(part.square_id,np.arange(n))
            variants.append((n,part[['x','y','angle']].to_numpy(float)))
        for vi,(n,p) in enumerate(variants):
            if side(p)<64:
                p=normalize(p,0);digest=hashlib.sha256(np.round(p,7).tobytes()).hexdigest()
                assert min_gap(p)>-1e-8,(n,vi,'invalid large donor')
                if digest not in seen:seen.add(digest);donors.append((p,'Couzo big:'+str(n)+':variant'+str(vi)))
    return base,sources,donors

def crop_data(p):
    e=.5*(abs(np.cos(p[:,2]))+abs(np.sin(p[:,2])))
    upper=cp.asarray(p[:,:2]+e[:,None]);wh=cp.asarray([(w,h) for w in range(1,32) for h in range(1,32)],dtype=cp.float64)
    r=cp.maximum(cp.maximum(upper[None,:,0]-wh[:,None,0],upper[None,:,1]-wh[:,None,1]),0)
    return r,cp.sort(r,axis=1)

def stitch(chain,gadgets,oriented,box=None):
    width=sum(gadgets[i]['w'] for i,m in chain)+sum(m==1 for i,m in chain[1:])
    height=sum(gadgets[i]['h'] for i,m in chain)+sum(m==2 for i,m in chain[1:])
    k=max(width,height);bx,by=(k,k) if box is None else box
    assert width<=bx and height<=by
    f=max(gadgets[i]['f'] for i,m in chain);placed=[];blocks=[];ox=0;oy=height-gadgets[chain[0][0]]['h']
    spans=[]
    for t,(gi,mode) in enumerate(chain):
        g=gadgets[gi];w,h=g['w'],g['h']
        if t:
            prev=gadgets[chain[t-1][0]];ox+=prev['w']+(mode==1);oy-=h+(mode==2)
        p=oriented[g['donor']][0];e=.5*(abs(np.cos(p[:,2]))+abs(np.sin(p[:,2])))
        r=np.maximum.reduce((p[:,0]+e-w,p[:,1]+e-h,np.zeros(len(p))))
        order=np.argsort(r,kind='stable');q=p[order[:w*h+g['dm']]].copy()
        if g.get('padding')=='up':q[:,1]+=max(0.,f-g['f'])
        if g.get('padding')=='right':q[:,0]+=max(0.,f-g['f'])
        q[:,:2]+=[ox,oy]
        blocks.append(q);spans.append((ox,oy,w,h))
    for x in range(bx):
        inspan=next((s for s in spans if s[0]<=x<s[0]+s[2]),None)
        if inspan:_,y0,w,h=inspan
        else:
            before=[s for s in spans if s[0]>x];y0=(before[0][1]+before[0][3]) if before else 0;h=0
        for y in range(by):
            if inspan and y0<=y<y0+h:continue
            shift=0 if y<y0 else f
            placed.append([x+.5+shift,y+.5+shift,0.])
    p=np.vstack((np.array(placed).reshape(-1,3),*blocks));assert len(p)==bx*by+sum(gadgets[i]['dm'] for i,m in chain)
    # Numerical clearance between otherwise exactly touching grid cells.
    p=normalize(p,0);p[:,:2]*=1+2e-10;return normalize(p,0),k

def run(seconds=240,limit=None,tag='fault_cuda',own_only=False,
        baseline_path=None,extra_donors=None,transpose=False,cushion=False):
    start=time.perf_counter();base,sources,donors=inputs()
    if baseline_path:
        base=read_csv(baseline_path)
        for n in range(1,1001):base.setdefault(n,grid(n))
        donors=[(p,'current certified portfolio:'+str(n)) for n,p in base.items()
                if len(p)>=5 and side(p)<math.ceil(math.sqrt(n))-1e-8]+donors
    if extra_donors:
        with np.load(extra_donors) as bank:
            donors=[(bank[k],'rectangular CUDA:'+k) for k in bank.files]+donors
    if own_only:donors=[d for d in donors if not d[1].startswith(('alex:','ensemble:'))]
    if limit:donors=donors[:limit]
    print(json.dumps(dict(stage='inputs',donors=len(donors),baseline_score=sum(side(p)/math.sqrt(n)-1 for n,p in base.items()))),flush=True)
    oriented=[]
    for p,label in donors:
        for trans in ([False,True] if transpose else [False]):
            r=p.copy()
            if trans:r[:,:2]=r[:,[1,0]];r[:,2]=np.pi/2-r[:,2]
            for sx,sy in [(1,1),(-1,1),(1,-1),(-1,-1)]:
                q=r.copy();q[:,0]*=sx;q[:,1]*=sy;q[:,2]*=sx*sy;q=normalize(q,0);oriented.append((q,label+f':transpose={trans}:{sx},{sy}'))
    best=cp.full(32*32*64*4,np.iinfo(np.uint64).max,cp.uint64);extract=cp.RawKernel(CUSHION_CUDA if cushion else CUDA,'extract_cushion' if cushion else 'extract');dag=cp.RawKernel(CUDA,'dag')
    ev1,ev2=cp.cuda.Event(),cp.cuda.Event();gpu=0.;done=0
    for di,(p,label) in enumerate(oriented):
        ev1.record();r,sorted_r=crop_data(p);extract(((31*31*63+127)//128,),(128,),(cp.asarray(p),np.int32(len(p)),r,sorted_r,np.int32(di),best));ev2.record();ev2.synchronize();gpu+=cp.cuda.get_elapsed_time(ev1,ev2)/1000;done=di+1
        if done%100==0:print(json.dumps(dict(stage='extract',oriented_done=done,seconds=time.perf_counter()-start,gpu_seconds=gpu)),flush=True)
        if time.perf_counter()-start>seconds*.65:break
    keys=cp.asnumpy(best).reshape(32,32,64,4);gadgets=[]
    for w,h,dm,flags in np.argwhere(keys<np.iinfo(np.uint64).max):
        key=int(keys[w,h,dm,flags]);di,z=divmod(key&0xffffffff,31*31*63)
        gadget=dict(w=int(w),h=int(h),dm=int(dm),br=int(flags//2),tl=int(flags%2),f=(key>>32)*1e-9,donor=di)
        if cushion:gadget['padding']='up' if flags==2 else 'right' if flags==1 else 'none'
        gadgets.append(gadget)
    print(json.dumps(dict(stage='dp',gadgets=len(gadgets),seconds=time.perf_counter()-start)),flush=True)
    g=cp.asarray([[r[k] for k in ['w','h','dm','br','tl']] for r in gadgets],dtype=cp.int32);f=cp.asarray([r['f'] for r in gadgets]);size=32*32*64*2
    dp=cp.full(size,2.,cp.float64);pg=cp.full(size,-1,cp.int32);ps=cp.full(size,-1,cp.int32);pm=cp.full(size,-1,cp.int32)
    ev1.record()
    for s in range(2,63):dag(((31*63*2+127)//128,),(128,),(np.int32(s),g,f,np.int32(len(gadgets)),dp,pg,ps,pm))
    ev2.record();ev2.synchronize();gpu+=cp.cuda.get_elapsed_time(ev1,ev2)/1000
    values=cp.asnumpy(dp).reshape(32,32,64,2);pga,psa,pma=[cp.asnumpy(a) for a in [pg,ps,pm]];candidates={};records=[];bad=[]
    def recover(ix):
        rev=[]
        while ix>=0:
            gi=int(pga[ix]);assert gi>=0;rev.append((gi,int(pma[ix])));ix=int(psa[ix])
        return list(reversed(rev))
    for n in range(5,1001):
        k=math.isqrt(n);dm=n-k*k
        if dm==0:continue
        slab=values[1:k+1,1:k+1,dm,:];j=np.unravel_index(np.argmin(slab),slab.shape);f0=float(slab[j])
        if k+f0>=side(base[n])-1e-8:continue
        w,h,b=j[0]+1,j[1]+1,j[2];ix=(((w*32+h)*64+dm)*2+b);rev=[]
        while ix>=0:
            gi=int(pga[ix]);assert gi>=0;rev.append((gi,int(pma[ix])));ix=int(psa[ix])
        chain=list(reversed(rev));p,kk=stitch(chain,gadgets,oriented)
        # If the used rectangle is smaller than k, add ordinary right/top grid borders.
        if kk<k:
            from packing_pipeline import border_expand
            p=border_expand(p,k-kk)
        assert len(p)==n,(n,len(p),kk,k,chain)
        gap=min_gap(p)
        if gap< -1e-9:bad.append(dict(n=n,gap=gap,chain=chain));continue
        candidates[n]=p;records.append(dict(n=n,before=side(base[n]),after=side(p),gain=(side(base[n])-side(p))/math.sqrt(n),chain=[dict(**gadgets[i],mode=m,origin=oriented[gadgets[i]['donor']][1]) for i,m in chain],min_gap=gap,structure='one_fault'))
    # Independent parallel faults: each horizontal strip carries its own chain.
    # Overhangs add vertically but only their maximum is needed horizontally.
    ev1.record();cost=dp.reshape(32,32,64,2).min(axis=3).copy()
    # Explicit GPU prefix minima; CuPy ufunc minimum.accumulate is unsupported.
    for z in range(1,32):cost[z]=cp.minimum(cost[z],cost[z-1])
    for z in range(1,32):cost[:,z]=cp.minimum(cost[:,z],cost[:,z-1])
    val=cp.full(32*32*64,2.,cp.float64);ph=cp.full(val.shape,-1,cp.int32);pd_=cp.full(val.shape,-1,cp.int32)
    val.reshape(32,32,64)[:,0,0]=0.;strip=cp.RawKernel(CUDA,'strips')
    for hh in range(1,32):strip(((31*64+127)//128,),(128,),(np.int32(hh),cost,val,ph,pd_))
    ev2.record();ev2.synchronize();gpu+=cp.cuda.get_elapsed_time(ev1,ev2)/1000
    va,ha,da=[cp.asnumpy(a).reshape(32,32,64) for a in [val,ph,pd_]];strip_records=[]
    for n in range(5,1001):
        k=math.isqrt(n);d=n-k*k
        if not d or k+va[k,k,d]>=side(candidates.get(n,base[n]))-1e-7:continue
        pieces=[];hh=k;dd=d;offset=0.;chains=[]
        while hh>0:
            h=int(ha[k,hh,dd]);t=int(da[k,hh,dd]);assert h>0 and t>=0
            if t:
                slab=values[1:k+1,1:h+1,t,:];jj=np.unravel_index(np.argmin(slab),slab.shape);w0,h0,b=jj[0]+1,jj[1]+1,jj[2]
                chain=recover((((w0*32+h0)*64+t)*2+b));p,_=stitch(chain,gadgets,oriented,box=(k,h));f0=max(gadgets[i]['f'] for i,m in chain)
                chains.append(dict(height=h,surplus=t,f=f0,chain=[dict(**gadgets[i],mode=m,origin=oriented[gadgets[i]['donor']][1]) for i,m in chain]))
            else:p=np.array([[x+.5,y+.5,0.] for x in range(k) for y in range(h)]);f0=0.
            p[:,1]+=offset;offset+=h+f0+1e-8;pieces.append(p);hh-=h;dd-=t
        p=normalize(np.vstack(pieces),0);assert len(p)==n
        gap=min_gap(p)
        if gap< -1e-9:bad.append(dict(n=n,gap=gap,structure='parallel_faults'));continue
        if side(p)>=side(candidates.get(n,base[n]))-1e-8:continue
        candidates[n]=p;strip_records.append(dict(n=n,before=side(base[n]),after=side(p),gain=(side(base[n])-side(p))/math.sqrt(n),strips=chains,min_gap=gap,structure='parallel_faults'))
    replaced={r['n'] for r in strip_records};records=[r for r in records if r['n'] not in replaced]+strip_records
    np.savez_compressed('artifacts/'+tag+'_packings.npz',**{str(n):p for n,p in candidates.items()})
    merged={**base,**candidates};export('artifacts/'+tag+'_submission.csv',merged,margin=0)
    report=dict(seconds=time.perf_counter()-start,cuda_seconds=gpu,baseline=baseline_path,extra_donors=extra_donors,transpose=transpose,cushion=cushion,oriented_done=done,oriented_planned=len(oriented),gadgets=len(gadgets),parallel_fault_wins=len(strip_records),improved=len(records),gain=sum(r['gain'] for r in records),score_estimate=sum(side(p)/math.sqrt(n)-1 for n,p in merged.items()),bad=bad,records=records,source_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest())
    Path('research/'+tag+'.json').write_text(json.dumps(report,indent=2));print(json.dumps({k:v for k,v in report.items() if k not in ['records','bad']}),flush=True)

if __name__=='__main__':
    a=argparse.ArgumentParser();a.add_argument('--seconds',type=float,default=240);a.add_argument('--limit',type=int);a.add_argument('--tag',default='fault_cuda');a.add_argument('--own-only',action='store_true');a.add_argument('--baseline');a.add_argument('--extra-donors');a.add_argument('--transpose',action='store_true');a.add_argument('--cushion',action='store_true');args=a.parse_args();run(args.seconds,args.limit,args.tag,args.own_only,args.baseline,args.extra_donors,args.transpose,args.cushion)


### gpu_seam_profiles.py

In [ ]:
%%writefile gpu_seam_profiles.py
"""CUDA fault composition with conservative separating-line tip profiles.

Deotte's corner-free test is replaced by quantized support bounds of clipped
corner polygons. A zero-buffer join is valid even when both tips are occupied
if one common diagonal line separates their actual occupied portions.
All crop selection, profile generation and minimax DAG search run on CUDA.
"""
import argparse,hashlib,json,math,time
from pathlib import Path
import cupy as cp
import numpy as np
from gpu_fault_composition import inputs,crop_data,stitch,read_csv
from solver import side,normalize
from gpu_families import min_gap
from hybrid_search import export

CUDA=r'''
__device__ int clip(int n,double*x,double*y,int axis,double bound,int sign){
 double xx[12],yy[12];int m=0;if(!n)return 0;
 for(int i=0;i<n;i++){
  int j=(i+1)%n;double a=sign*((axis?y[i]:x[i])-bound),b=sign*((axis?y[j]:x[j])-bound);
  if(a>=0){xx[m]=x[i];yy[m++]=y[i];}
  if((a<0)!=(b<0)){double t=a/(a-b);xx[m]=x[i]+t*(x[j]-x[i]);yy[m++]=y[i]+t*(y[j]-y[i]);}
 }
 for(int i=0;i<m;i++){x[i]=xx[i];y[i]=yy[i];}return m;
}
__device__ void bounds(double x,double y,double a,double c,double s,double*lo,double*hi){
 double co=.5*cos(a),si=.5*sin(a),xx[12]={x+co-si,x-co-si,x-co+si,x+co+si},yy[12]={y+si+co,y-si+co,y-si-co,y+si-co};
 int n=4;n=clip(n,xx,yy,0,-2e-8,1);n=clip(n,xx,yy,0,1.00000002,-1);n=clip(n,xx,yy,1,-2e-8,1);n=clip(n,xx,yy,1,1.00000002,-1);
 for(int i=0;i<n;i++){double v=c*xx[i]-s*yy[i];*lo=fmin(*lo,v);*hi=fmax(*hi,v);}
}
extern "C" __global__ void extract(const double*p,int n,const double*r,const double*sorted,int donor,double c,double s,unsigned long long*best){
 int z=blockDim.x*blockIdx.x+threadIdx.x;if(z>=31*31*63)return;
 int dm=z%63+1,t=z/63,w=t/31+1,h=t%31+1,m=w*h+dm;if(m>n)return;
 double f=sorted[t*n+m-1]+3e-8;if(f<=3e-8||f>=.999999)return;
 double br=-2.,tl=2.;int count=0;
 for(int i=0;i<n;i++)if(r[t*n+i]<=f-2e-8){
  count++;double x=p[3*i],y=p[3*i+1],a=p[3*i+2],e=.5*(fabs(cos(a))+fabs(sin(a))),dummy;
  if(x+e>w-2e-8&&y-e<1.00000002){dummy=2.;bounds(x-w,y,a,c,s,&dummy,&br);}
  if(x-e<1.00000002&&y+e>h-2e-8){dummy=-2.;bounds(x,y-h,a,c,s,&tl,&dummy);}
 }
 if(count<m)return;
 int b=br < -1.5?0:max(0,min(32,(int)ceil((br+1)*16+2e-6)));
 int tbin=tl>1.5?32:max(0,min(32,(int)floor((tl+1)*16-2e-6)));
 unsigned long long q=(unsigned long long)ceil(f*1e9),job=donor*(31*31*63)+z;
 atomicMin(best+(((w*32+h)*64+dm)*33+b)*33+tbin,(q<<32)|job);
}
extern "C" __global__ void dag(int sum,const int*g,const unsigned int*f,int ng,unsigned long long*dp,const unsigned long long*prefix){
 int z=blockIdx.x,w=z/63+1,d=z%63+1,h=sum-w;if(h<1||h>31)return;
 for(int i=threadIdx.x;i<ng;i+=blockDim.x){
  int gw=g[5*i],gh=g[5*i+1],gd=g[5*i+2],br=g[5*i+3],tl=g[5*i+4];if(gw>w||gh>h||gd>d)continue;
  unsigned long long*dst=dp+(((w*32+h)*64+d)*33+br);unsigned int limit=(*dst)>>32;
  if(f[i]>=limit)continue;
  if(gw==w&&gh==h&&gd==d){atomicMin(dst,((unsigned long long)f[i]<<32)|(4*i+3));continue;}
  if(gd==d)continue;
  for(int mode=0;mode<3;mode++){
   int x=w-gw-(mode==1),y=h-gh-(mode==2);if(x<1||y<1)continue;
   unsigned int prev=prefix[(((x*32+y)*64+d-gd)*33+(mode?32:tl))]>>32;
   unsigned int q=max(prev,f[i]);if(q>=limit)continue;
   atomicMin(dst,((unsigned long long)q<<32)|(4*i+mode));
  }
 }
}
extern "C" __global__ void scan(int sum,const unsigned long long*dp,unsigned long long*prefix){
 int z=blockIdx.x*blockDim.x+threadIdx.x;if(z>=31*63)return;
 int w=z/63+1,d=z%63+1,h=sum-w;if(h<1||h>31)return;
 unsigned long long best=~0ULL;
 for(int b=0;b<33;b++){
  int ix=(((w*32+h)*64+d)*33+b);unsigned long long q=((dp[ix]>>32)<<32)|b;best=min(best,q);prefix[ix]=best;
 }
}
'''

def run(degrees=45.,seconds=240,tag='seam_cuda',limit=None,bins=32,interior=False,
        baseline_path=None,extra_donors=None,transpose=False):
    start=time.perf_counter();base,sources,donors=inputs()
    if baseline_path:
        from solver import grid
        base=read_csv(baseline_path)
        for n in range(1,1001):base.setdefault(n,grid(n))
        donors=[(p,'current certified portfolio:'+str(n)) for n,p in base.items()
                if len(p)>=5 and side(p)<math.ceil(math.sqrt(n))-1e-8]+donors
    else:
        for n,p in read_csv('research/latest/kirill.csv').items():
            if side(p)<side(base[n]):base[n]=p;sources[n]='Kirill V6'
    if extra_donors:
        with np.load(extra_donors) as bank:
            donors.extend((bank[k],'rectangular CUDA:'+k) for k in bank.files)
    if limit:donors=donors[:limit]
    if interior:
        # A crop can begin at an interior contact, not only a container corner.
        # Every retained unit square must lie wholly above/right of that anchor.
        # Re-normalization removes empty slack at the new crop origin.
        extra=[];seen=set()
        for p,label in donors:
            if len(p)<200:continue
            e=.5*(abs(np.cos(p[:,2]))+abs(np.sin(p[:,2])));lo=p[:,:2]-e[:,None];hi=p[:,:2]+e[:,None];span=hi.max(0)
            for fx,fy in [(0,.23),(.23,0),(.23,.23),(.47,.13),(.13,.47)]:
                ox=0. if fx==0 else float(np.sort(lo[:,0])[int(fx*len(p))]);oy=0. if fy==0 else float(np.sort(lo[:,1])[int(fy*len(p))]);keep=(lo[:,0]>=ox-1e-10)&(lo[:,1]>=oy-1e-10)
                if keep.sum()<5:continue
                q=normalize(p[keep],0);digest=hashlib.sha256(np.round(q,7).tobytes()).hexdigest()
                if digest in seen:continue
                seen.add(digest);extra.append((q,label+f':interior-{fx},{fy}'))
        print(json.dumps(dict(stage='interior crop sources',extra=len(extra))),flush=True);donors+=extra
    oriented=[]
    for p,label in donors:
        for trans in ([False,True] if transpose else [False]):
            r=p.copy()
            if trans:r[:,:2]=r[:,[1,0]];r[:,2]=np.pi/2-r[:,2]
            for sx,sy in [(1,1),(-1,1),(1,-1),(-1,-1)]:
                q=r.copy();q[:,0]*=sx;q[:,1]*=sy;q[:,2]*=sx*sy;oriented.append((normalize(q,0),label+f':transpose={trans}:{sx},{sy}'))
    nb=bins+1
    src=CUDA.replace('33','NB').replace('min(32,','min(NB-1,').replace('tl>1.5?32','tl>1.5?NB-1').replace('(mode?32:tl)','(mode?NB-1:tl)').replace('*16+','*((NB-1)/2.)+').replace('*16-','*((NB-1)/2.)-')
    mod=cp.RawModule(code='#define NB '+str(nb)+'\n'+src,name_expressions=['extract','dag','scan']);extract=mod.get_function('extract');dag=mod.get_function('dag');scan=mod.get_function('scan')
    best=cp.full(32*32*64*nb*nb,np.iinfo(np.uint64).max,cp.uint64);c,s=math.cos(math.radians(degrees)),math.sin(math.radians(degrees));a,b=cp.cuda.Event(),cp.cuda.Event();gpu=0.;done=0
    print(json.dumps(dict(stage='profile extraction',degrees=degrees,donors=len(donors))),flush=True)
    for di,(p,label) in enumerate(oriented):
        a.record();r,sr=crop_data(p);extract(((31*31*63+127)//128,),(128,),(cp.asarray(p),np.int32(len(p)),r,sr,np.int32(di),np.float64(c),np.float64(s),best));b.record();b.synchronize();gpu+=cp.cuda.get_elapsed_time(a,b)/1000;done=di+1
        if done%500==0:print(json.dumps(dict(stage='extract',done=done,seconds=time.perf_counter()-start,cuda_seconds=gpu)),flush=True)
        if time.perf_counter()-start>seconds*.6:break
    # Only populated profile entries cross PCIe, never the dense 570 MB table.
    active=cp.flatnonzero(best!=np.iinfo(np.uint64).max);idx=cp.asnumpy(active);keys=cp.asnumpy(best[active]);del best,active
    gadgets=[]
    for ix,key in zip(idx,keys):
        ix=int(ix);tl=ix%nb;ix//=nb;br=ix%nb;ix//=nb;dm=ix%64;ix//=64;h=ix%32;w=ix//32;key=int(key);di,_=divmod(key&0xffffffff,31*31*63)
        gadgets.append(dict(w=w,h=h,dm=dm,br=br,tl=tl,f=(key>>32)*1e-9,donor=di))
    gadgets.sort(key=lambda g:g['f']);ng=len(gadgets);g=cp.asarray([[r[k] for k in ['w','h','dm','br','tl']] for r in gadgets],cp.int32);f=cp.asarray([round(r['f']*1e9) for r in gadgets],cp.uint32)
    print(json.dumps(dict(stage='profile DP',gadgets=ng,seconds=time.perf_counter()-start)),flush=True)
    dp=cp.full(32*32*64*nb,np.iinfo(np.uint64).max,cp.uint64);prefix=cp.full_like(dp,np.iinfo(np.uint64).max);a.record()
    for total in range(2,63):
        dag((31*63,),(128,),(np.int32(total),g,f,np.int32(ng),dp,prefix));scan(((31*63+127)//128,),(128,),(np.int32(total),dp,prefix))
    b.record();b.synchronize();gpu+=cp.cuda.get_elapsed_time(a,b)/1000;keys=cp.asnumpy(dp).reshape(32,32,64,nb);pre=cp.asnumpy(prefix).reshape(keys.shape)
    candidates={};records=[];bad=[]
    for n in range(5,1001):
        k=math.isqrt(n);dm=n-k*k
        if not dm:continue
        slab=keys[1:k+1,1:k+1,dm,:];jj=np.unravel_index(np.argmin(slab>>32),slab.shape);w,h,br=jj[0]+1,jj[1]+1,jj[2];value=int(slab[jj])>>32
        if k+value*1e-9>=side(base[n])-1e-7:continue
        rev=[];dd=dm
        while True:
            key=int(keys[w,h,dd,br]);action=key&0xffffffff;gi,mode=divmod(action,4);rec=gadgets[gi];rev.append((gi,mode))
            if mode==3:break
            w-=rec['w']+(mode==1);h-=rec['h']+(mode==2);dd-=rec['dm'];br=int(pre[w,h,dd,bins if mode else rec['tl']])&0xffffffff
        chain=list(reversed(rev));p,kk=stitch(chain,gadgets,oriented)
        if kk<k:
            from packing_pipeline import border_expand
            p=border_expand(p,k-kk)
        assert len(p)==n
        gap=min_gap(p)
        if gap< -1e-9:
            bad.append(dict(n=n,gap=gap,value=value,w=w,h=h,chain=[gadgets[i] for i,m in chain] if len(bad)<3 else []));continue
        candidates[n]=p;records.append(dict(n=n,before=side(base[n]),after=side(p),gain=(side(base[n])-side(p))/math.sqrt(n),min_gap=gap,chain=[dict(**gadgets[i],mode=m,origin=oriented[gadgets[i]['donor']][1]) for i,m in chain]))
    np.savez_compressed('artifacts/'+tag+'_packings.npz',**{str(n):p for n,p in candidates.items()});merged={**base,**candidates};export('artifacts/'+tag+'_submission.csv',merged,margin=0)
    report=dict(seconds=time.perf_counter()-start,cuda_seconds=gpu,degrees=degrees,bins=bins,interior_crops=interior,baseline=baseline_path,extra_donors=extra_donors,transpose=transpose,oriented_done=done,oriented_planned=len(oriented),gadgets=ng,improved=len(records),gain=sum(r['gain'] for r in records),score_estimate=sum(side(p)/math.sqrt(n)-1 for n,p in merged.items()),bad=bad,records=records,source_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest())
    Path('research/'+tag+'.json').write_text(json.dumps(report,indent=2));print(json.dumps({k:v for k,v in report.items() if k not in ['records','bad']}),flush=True)

if __name__=='__main__':
    ap=argparse.ArgumentParser();ap.add_argument('--degrees',type=float,default=45);ap.add_argument('--seconds',type=float,default=240);ap.add_argument('--tag',default='seam_cuda');ap.add_argument('--limit',type=int);ap.add_argument('--bins',type=int,default=32);ap.add_argument('--interior',action='store_true');ap.add_argument('--baseline');ap.add_argument('--extra-donors');ap.add_argument('--transpose',action='store_true');a=ap.parse_args();run(a.degrees,a.seconds,a.tag,a.limit,a.bins,a.interior,a.baseline,a.extra_donors,a.transpose)


### gpu_adaptive_seams.py

In [ ]:
%%writefile gpu_adaptive_seams.py
"""CUDA minimax packing composition with a separate separator at every join.

Each block has an incoming and an outgoing support profile. They need not use
the same line normal. Prefix minima remove dominated (f,BR,TL) states on GPU;
only the compact frontier crosses PCIe. Diagonal composition geometry is
credited to avazaku / Chris Deotte; extends our CUDA corner-support solver.
"""
import argparse,hashlib,json,math,time
from pathlib import Path
import cupy as cp
import numpy as np
from gpu_fault_composition import inputs,crop_data,stitch,read_csv
from gpu_seam_profiles import CUDA as OLD_CUDA
from solver import normalize,side,grid
from gpu_families import min_gap

HELPERS=OLD_CUDA[:OLD_CUDA.index('extern "C" __global__ void extract')]
CUDA=r'''
extern "C" __global__ void extract(const double*p,int n,const double*r,
 const double*sorted,int donor,const double*cs,unsigned long long*best){
 int z=blockDim.x*blockIdx.x+threadIdx.x;if(z>=31*31*63)return;
 int dm=z%63+1,t=z/63,w=t/31+1,h=t%31+1,m=w*h+dm;if(m>n)return;
 double f=sorted[t*n+m-1]+3e-8;if(f<=3e-8||f>=.999999)return;
 double br[NA],tl[NA];for(int a=0;a<NA;a++){br[a]=-2.;tl[a]=2.;}
 int count=0;
 for(int i=0;i<n;i++)if(r[t*n+i]<=f-2e-8){
  count++;double x=p[3*i],y=p[3*i+1],ang=p[3*i+2],e=.5*(fabs(cos(ang))+fabs(sin(ang)));
  for(int a=0;a<NA;a++){
   double dummy,c=cs[2*a],s=cs[2*a+1];
   if(x+e>w-2e-8&&y-e<1.00000002){dummy=2.;bounds(x-w,y,ang,c,s,&dummy,&br[a]);}
   if(x-e<1.00000002&&y+e>h-2e-8){dummy=-2.;bounds(x,y-h,ang,c,s,&tl[a],&dummy);}
  }
 }
 if(count<m)return;
 unsigned long long q=(unsigned long long)ceil(f*1e9),job=donor*(31*31*63)+z,key=(q<<32)|job;
 for(int ai=0;ai<NA;ai++)for(int ao=0;ao<NA;ao++){
  int b=br[ao]<-1.5?0:max(0,min(NB-1,(int)ceil((br[ao]+1)*((NB-1)/2.)+2e-6)));
  int tt=tl[ai]>1.5?NB-1:max(0,min(NB-1,(int)floor((tl[ai]+1)*((NB-1)/2.)-2e-6)));
  long long ix=((((((long long)w*32+h)*64+dm)*NA+ai)*NA+ao)*NB+b)*NB+tt;
  atomicMin(best+ix,key);
 }
}
extern "C" __global__ void dag(int sum,const int*g,const unsigned int*f,int ng,
 unsigned long long*dp,const unsigned long long*prefix){
 int z=blockIdx.x,w=z/63+1,d=z%63+1,h=sum-w;if(h<1||h>31)return;
 for(int i=threadIdx.x;i<ng;i+=blockDim.x){
  const int* row=g+7*i;int gw=row[0],gh=row[1],gd=row[2],br=row[3],tl=row[4],ai=row[5],ao=row[6];
  if(gw>w||gh>h||gd>d)continue;
  unsigned long long*dst=dp+((((w*32+h)*64+d)*NA+ao)*NB+br);
  unsigned int limit=(*dst)>>32;if(f[i]>=limit)continue;
  if(gw==w&&gh==h&&gd==d){atomicMin(dst,((unsigned long long)f[i]<<32)|(4*i+3));continue;}
  if(gd==d)continue;
  for(int mode=0;mode<3;mode++){
   int x=w-gw-(mode==1),y=h-gh-(mode==2);if(x<1||y<1)continue;
   unsigned int prev=prefix[((((x*32+y)*64+d-gd)*NA+ai)*NB+(mode?NB-1:tl))]>>32;
   unsigned int q=max(prev,f[i]);if(q>=limit)continue;
   atomicMin(dst,((unsigned long long)q<<32)|(4*i+mode));
  }
 }
}
extern "C" __global__ void scan(int sum,const unsigned long long*dp,unsigned long long*prefix){
 int z=blockDim.x*blockIdx.x+threadIdx.x;if(z>=31*63*NA)return;
 int ao=z%NA;z/=NA;int w=z/63+1,d=z%63+1,h=sum-w;if(h<1||h>31)return;
 unsigned long long best=~0ULL;
 for(int br=0;br<NB;br++){
  int ix=((((w*32+h)*64+d)*NA+ao)*NB+br);
  best=min(best,((dp[ix]>>32)<<32)|br);prefix[ix]=best;
 }
}
'''

def pareto(best,nb):
    """Smaller BR and larger TL are better; retain incomparable f profiles."""
    sentinel=np.uint64(0xffffffffffffffff)
    active=cp.flatnonzero(best!=sentinel);original=best.ravel()[active].copy()
    shape=best.shape;front=best.reshape(-1,nb,nb)
    for b in range(1,nb):front[:,b,:]=cp.minimum(front[:,b,:],front[:,b-1,:])
    for t in range(nb-2,-1,-1):front[:,:,t]=cp.minimum(front[:,:,t],front[:,:,t+1])
    br=active//nb%nb;tl=active%nb;flat=front.ravel()
    left=cp.where(br>0,flat[cp.maximum(active-nb,0)],np.uint64(0xffffffffffffffff))
    upper=cp.where(tl<nb-1,flat[cp.minimum(active+1,flat.size-1)],np.uint64(0xffffffffffffffff))
    keep=(cp.minimum(left,upper)>>32)>(original>>32)
    return cp.asnumpy(active[keep]),cp.asnumpy(original[keep]),int(active.size)

def run(baseline,tag='adaptive_seams1',seconds=480,bins=16,degrees=(30,45,60),limit=None,audit=False):
    start=time.perf_counter();NA=len(degrees);NB=bins+1
    free,_=cp.cuda.runtime.memGetInfo();pool_limit=min(8_000_000_000,int(free)-1_000_000_000)
    table_size=32*32*64*NA*NA*NB*NB*8
    if table_size+500_000_000>pool_limit:raise RuntimeError('Not enough shared GPU headroom for requested profile table')
    cp.get_default_memory_pool().set_limit(size=pool_limit)
    labels=json.loads(Path('artifacts/fault_source_labels.json').read_text(encoding='utf8'))
    with np.load('artifacts/fault_source_donors.npz')as frozen:legacy=[(frozen[k],labels[k])for k in frozen.files]
    base=read_csv(baseline)
    donors=[(p,'Public frontier:'+str(n))for n,p in base.items()if len(p)>4]
    for n in range(1,1001):base.setdefault(n,grid(n))
    fresh=[]
    for path in ['artifacts/tip_frontier1_gadgets.npz','artifacts/crystal_polished1_gadgets.npz']:
        with np.load(path)as z:fresh.extend((z[k],'Own fresh crystal:'+k)for k in z.files)
    donors=fresh+donors+legacy
    if limit:donors=donors[:limit]
    oriented=[]
    for p,label in donors:
        for trans in [False,True]:
            r=p.copy()
            if trans:r[:,:2]=r[:,[1,0]];r[:,2]=np.pi/2-r[:,2]
            for sx,sy in [(1,1),(-1,1),(1,-1),(-1,-1)]:
                q=r.copy();q[:,0]*=sx;q[:,1]*=sy;q[:,2]*=sx*sy;oriented.append((normalize(q,0),label+f':T={trans}:{sx},{sy}'))
    src=f'#define NA {NA}\n#define NB {NB}\n'+HELPERS+CUDA
    mod=cp.RawModule(code=src,name_expressions=['extract','dag','scan']);extract,dag,scan=[mod.get_function(k)for k in ['extract','dag','scan']]
    best=cp.full((32,32,64,NA,NA,NB,NB),np.iinfo(np.uint64).max,cp.uint64)
    cs=cp.asarray([[math.cos(math.radians(d)),math.sin(math.radians(d))]for d in degrees])
    a,b=cp.cuda.Event(),cp.cuda.Event();gpu=0.;done=0
    print(json.dumps(dict(stage='adaptive extraction',degrees=degrees,bins=bins,oriented=len(oriented),table_bytes=table_size,pool_limit=pool_limit)),flush=True)
    for di,(p,label)in enumerate(oriented):
        a.record();r,sr=crop_data(p);extract(((31*31*63+127)//128,),(128,),(cp.asarray(p),np.int32(len(p)),r,sr,np.int32(di),cs,best))
        b.record();b.synchronize();gpu+=cp.cuda.get_elapsed_time(a,b)/1000;done=di+1
        if done%500==0:print(json.dumps(dict(stage='extract',done=done,seconds=time.perf_counter()-start,cuda_seconds=gpu)),flush=True)
        if time.perf_counter()-start>seconds*.7:break
    a.record();idx,keys,unpruned=pareto(best,NB);b.record();b.synchronize();gpu+=cp.cuda.get_elapsed_time(a,b)/1000
    del best,r,sr;cp.get_default_memory_pool().free_all_blocks()
    tl=idx%NB;z=idx//NB;br=z%NB;z//=NB;ao=z%NA;z//=NA;ai=z%NA;z//=NA;dm=z%64;z//=64;h=z%32;w=z//32
    f=(keys>>32).astype(np.uint32);donor=((keys&np.uint64(0xffffffff))//(31*31*63)).astype(np.int32)
    gg=np.column_stack((w,h,dm,br,tl,ai,ao)).astype(np.int32);order=np.argsort(f,kind='stable');gg,f,donor=gg[order],f[order],donor[order]
    ng=len(f);g=cp.asarray(gg);cf=cp.asarray(f)
    print(json.dumps(dict(stage='adaptive DP',populated=unpruned,pareto=ng,seconds=time.perf_counter()-start)),flush=True)
    dp=cp.full((32,32,64,NA,NB),np.iinfo(np.uint64).max,cp.uint64);prefix=cp.full_like(dp,np.iinfo(np.uint64).max)
    a.record()
    for total in range(2,63):
        dag((31*63,),(128,),(np.int32(total),g,cf,np.int32(ng),dp,prefix))
        scan(((31*63*NA+127)//128,),(128,),(np.int32(total),dp,prefix))
    b.record();b.synchronize();gpu+=cp.cuda.get_elapsed_time(a,b)/1000;values,pre=map(cp.asnumpy,[dp,prefix])
    winners={};records=[];bad=[];audits=[];audit_bank={};leverage={}
    for n in range(5,1001):
        k=math.isqrt(n);d=n-k*k
        if not d:continue
        slab=values[1:k+1,1:k+1,d];jj=np.unravel_index(np.argmin(slab>>32),slab.shape);value=int(slab[jj])>>32
        if value==0xffffffff:continue
        improvement=k+value*1e-9<side(base[n])-1e-7
        if not improvement and not audit:continue
        w,h,ao,br=jj[0]+1,jj[1]+1,jj[2],jj[3];rev=[];dd=d
        while True:
            action=int(values[w,h,dd,ao,br])&0xffffffff;gi,mode=divmod(action,4);rev.append((gi,mode))
            if mode==3:break
            gw,gh,gd,gbr,tl,ai,_=map(int,gg[gi]);w-=gw+(mode==1);h-=gh+(mode==2);dd-=gd;ao=ai;br=int(pre[w,h,dd,ao,NB-1 if mode else tl])&0xffffffff
        chain=list(reversed(rev));gadgets=[];recipe=[]
        for gi,mode in chain:
            w,h,d,br,tl,ai,ao=map(int,gg[gi]);di=int(donor[gi]);rr=dict(w=w,h=h,dm=d,br=br,tl=tl,f=int(f[gi])*1e-9,donor=di)
            gadgets.append(rr);recipe.append(dict(**rr,mode=mode,incoming_degrees=degrees[ai],outgoing_degrees=degrees[ao],origin=oriented[di][1]))
        if audit:
            maximum=max(b['f']for b in recipe)
            active=[b for b in recipe if maximum-b['f']<2e-6]
            row=dict(n=n,baseline=side(base[n]),predicted=k+maximum,chain=recipe)
            audits.append(row)
            for block in active:
                identity=(block['w'],block['h'],block['dm'],round(block['f'],8),block['origin'])
                key=hashlib.sha256(repr(identity).encode()).hexdigest()[:14]
                exact_others=max([b['f']for b in recipe if b is not block]+[0.])
                gain=max(0.,side(base[n])-k-max(block['f']-.01,exact_others))/math.sqrt(n)
                joint=max(0.,side(base[n])-k-(maximum-.01))/math.sqrt(n)/len(active)
                if key not in leverage:
                    g=block;source=oriented[g['donor']][0];e=.5*(abs(np.cos(source[:,2]))+abs(np.sin(source[:,2])))
                    r=np.maximum.reduce((source[:,0]+e-g['w'],source[:,1]+e-g['h'],np.zeros(len(source))))
                    audit_bank[key]=source[np.argsort(r,kind='stable')[:g['w']*g['h']+g['dm']]].copy()
                    leverage[key]=dict(key=key,w=g['w'],h=g['h'],dm=g['dm'],f=g['f'],origin=g['origin'],counterfactual_gain=0.,joint_bound=0.,uses=[])
                leverage[key]['counterfactual_gain']+=gain;leverage[key]['joint_bound']+=joint;leverage[key]['uses'].append(n)
        if not improvement:continue
        p,kk=stitch([(j,mode)for j,(_,mode)in enumerate(chain)],gadgets,oriented)
        if kk<k:
            from packing_pipeline import border_expand
            p=border_expand(p,k-kk)
        assert len(p)==n,(n,len(p),recipe)
        gap=min_gap(p)
        if gap< -1e-9:bad.append(dict(n=n,gap=gap,recipe=recipe));continue
        winners[n]=p;records.append(dict(n=n,before=side(base[n]),after=side(p),gain=(side(base[n])-side(p))/math.sqrt(n),min_gap=gap,chain=recipe))
    np.savez_compressed(f'artifacts/{tag}_packings.npz',**{str(n):p for n,p in winners.items()})
    report=dict(baseline=baseline,seconds=time.perf_counter()-start,cuda_seconds=gpu,degrees=degrees,bins=bins,
        oriented_done=done,oriented_planned=len(oriented),populated_states=unpruned,pareto_states=ng,
        improved=len(winners),gain=sum(r['gain']for r in records),records=records,bad=bad,
        source_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest())
    Path(f'research/{tag}.json').write_text(json.dumps(report,indent=2),encoding='utf8')
    if audit:
        rank=sorted(leverage.values(),key=lambda r:-(r['counterfactual_gain']+.15*r['joint_bound']))
        np.savez_compressed(f'artifacts/{tag}_active_gadgets.npz',**audit_bank)
        Path(f'research/{tag}_bottlenecks.json').write_text(json.dumps(dict(ranked=rank,chains=audits),indent=2),encoding='utf8')
    print(json.dumps({k:v for k,v in report.items()if k not in ['records','bad']}),flush=True)

if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('--baseline',required=True);p.add_argument('--seconds',type=float,default=480);p.add_argument('--bins',type=int,default=16);p.add_argument('--degrees',type=float,nargs='+',default=[30,45,60]);p.add_argument('--limit',type=int);p.add_argument('--tag',default='adaptive_seams1');p.add_argument('--audit',action='store_true');a=p.parse_args();run(a.baseline,a.tag,a.seconds,a.bins,tuple(a.degrees),a.limit,a.audit)


### gpu_projected_union.py

In [ ]:
%%writefile gpu_projected_union.py
"""CUDA multi-start SAT proposals with feasibility restoration before rejection.

Kirill Tymchishin's Projected Union motivates admitting near-feasible states
through an LP before deciding whether they improve an incumbent. This uses
the existing independent CUDA C++ FP64 L-BFGS and rectangular LP helpers.
No downloaded optimizer code is executed. Every accepted witness passes the
unchanged official metric, including the ordinary pandas float64 replay.
"""
import os
for k in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS'):os.environ[k]='1'
import argparse,json,time,math,hashlib
from pathlib import Path
import numpy as np
import cupy as cp
from solver import normalize,side
from gpu_fault_composition import read_csv
from gpu_rect_optimizer import quench,check
from gpu_ftmc_refill import repair,rect_lp,certify,atom_json,atom_npz

def propose(p,seed,batch=32):
    rng=np.random.default_rng(seed);L=side(p);p=normalize(p,0)
    depth=np.resize([.0002,.001,.004,.012,.03,.06,.10,.16],batch)
    q=np.repeat(p[None],batch,axis=0);boxes=np.repeat((L-depth)[:,None],2,axis=1)
    for b in range(batch):
        q[b,:,:2]=.5+(p[:,:2]-.5)*(boxes[b,0]-1)/(L-1)
        mode=(b//8)%4
        if mode==1:
            # Move a spatial patch coherently; its contacts with the rest may switch.
            pivot=p[rng.integers(len(p)),:2];ids=np.argsort(((p[:,:2]-pivot)**2).sum(1))[:min(len(p),20)]
            delta=rng.choice([-1,1])*.025;c,s=np.cos(delta),np.sin(delta)
            q[b,ids,:2]=(q[b,ids,:2]-pivot)@np.array([[c,s],[-s,c]])+pivot
            q[b,ids,2]+=delta
        elif mode==2:
            # A narrow fault is displaced in opposite directions across its midline.
            a=rng.uniform(0,np.pi);u=np.array([np.cos(a),np.sin(a)]);v=np.array([-u[1],u[0]])
            proj=p[:,:2]@u;cut=np.median(proj);q[b,:,:2]+=np.sign(proj-cut)[:,None]*v*.035
            q[b,:,2]+=rng.normal(0,.008,len(p))
        elif mode==3:
            q[b,:,:2]+=rng.normal(0,.018,p[:,:2].shape);q[b,:,2]+=rng.normal(0,.012,len(p))
    return q,boxes

def run(baseline,tag,seconds=420,limit=80):
    start=time.perf_counter();checks=check();bank=read_csv(baseline);accepted={};history=[];cuda_seconds=0.
    targets=[]
    for n,p in bank.items():
        if n<40 or n>999 or side(p)>=math.ceil(math.sqrt(n))-1e-5:continue
        a=abs((p[:,2]+np.pi/4)%(np.pi/2)-np.pi/4);t=a[a>.002]
        if len(t) and np.max(abs(t-np.pi/4))>1e-4:targets.append(n)
    # Cover different sizes; prioritize the independently productive public cases.
    fresh=json.loads(Path('research/refresh-downloads/update25-kirilltym/experiment_logs/round22_summary.json').read_text())
    hot=[r['n']for r in sorted(fresh['changes'],key=lambda r:-r['gain'])]
    targets=list(dict.fromkeys(hot+sorted(targets,key=lambda n:1/math.sqrt(n),reverse=True)))[:limit]
    for idx,n in enumerate(targets):
        if time.perf_counter()-start>seconds-18:break
        p=bank[n];L0=side(p);q,boxes=propose(p,20261025+n)
        a,b=cp.cuda.Event(),cp.cuda.Event();a.record();q,stats=quench(q,boxes,steps=1400);b.record();b.synchronize();cuda_seconds+=cp.cuda.get_elapsed_time(a,b)/1000
        trials=cp.asnumpy(q);energies=cp.asnumpy(stats)[:,0];best=p;lp_wins=0;direct_wins=0
        # Limit CPU restoration to the best four distinct proposals per puzzle.
        for trial in trials[np.argsort(energies)[:4]]:
            direct=repair(trial)
            if direct is not None and side(direct)<side(best)-1e-7 and certify(direct):best=direct;direct_wins+=1
            # Unlike the old gate, overlap does not immediately discard the state.
            k=math.floor(L0);candidate=rect_lp(trial,k,k,rounds=2)
            if candidate is not None and side(candidate)<side(best)-1e-7 and certify(candidate):best=candidate;lp_wins+=1
        gain=(L0-side(best))/math.sqrt(n)
        if gain>1e-10:accepted[str(n)]=best;bank[n]=best
        history.append(dict(n=n,before=L0,after=side(best),gain=gain,minimum_energy=float(energies.min()),direct_wins=direct_wins,lp_wins=lp_wins,elapsed_seconds=time.perf_counter()-start))
        atom_npz(f'artifacts/{tag}_packings.npz',accepted)
        atom_json(f'research/{tag}.json',dict(baseline=baseline,checks=checks,batch=32,iterations=1400,seconds=time.perf_counter()-start,cuda_seconds=cuda_seconds,records=history,improved=len(accepted),gain=sum(r['gain']for r in history),source_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()))
        print(json.dumps(history[-1]),flush=True)

if __name__=='__main__':
    a=argparse.ArgumentParser();a.add_argument('--baseline',required=True);a.add_argument('--tag',default='update25_projected');a.add_argument('--seconds',type=float,default=420);a.add_argument('--limit',type=int,default=80);v=a.parse_args();run(v.baseline,v.tag,v.seconds,v.limit)


### research/official_metric.py

In [ ]:
%%writefile research/official_metric.py
"""Metric for the "Packing N Unit Squares into the Smallest Square" competition.

Participants submit packings of n unit squares (n=1..1000) inside a bounding
square.  Each square is specified by its center (x, y) and rotation angle.
Coordinates must be plain decimal literals, optionally in scientific notation
(e.g. ``2.70710678118654752440`` or ``1.5e-3``).  Symbolic / closed-form
expressions are not accepted -- submit their decimal expansion instead, at as
many digits as you like.

Scoring
-------
For each puzzle n the bounding side length L is measured
translation-invariantly as ``max(max_x - min_x, max_y - min_y)`` over all
corners.  The per-puzzle score is the excess over the area lower bound

    score(n) = L / sqrt(n) - 1

which is 0.0 exactly when the packing achieves the area lower bound (only
possible for perfect squares packed on the grid) and grows above 0 as the
bounding square wastes area.  **Lower is better**; total score is the sum
across n=1..N.

+-------------------+---------------------------------------------------------+
| Condition         | Score                                                   |
+===================+=========================================================+
| feasible packing  | L / sqrt(n) - 1     (>= 0 always)                       |
+-------------------+---------------------------------------------------------+
| infeasible/invalid| L_0 / sqrt(n) - 1   (grid ceiling — no worse than       |
|                   |                      submitting the trivial grid)       |
+-------------------+---------------------------------------------------------+

For reference: ``L_B = sqrt(n)`` is the area lower bound and
``L_0 = ceil(sqrt(n))`` is the trivial grid packing.  The best-known side
length plays no part -- beating a record just yields a smaller excess -- so
this metric reads no baseline values at all.

Feasibility is checked with interval arithmetic (mpmath) at escalating
precision to certify that no two squares overlap.

Kaggle entry point
------------------
``score(solution, submission, row_id_column_name)`` — called by the Kaggle
backend.  ``solution`` is a DataFrame loaded from ``solution.csv``; only its
``n`` column is read, to fix which puzzles are scored (and which absent ones
are charged the grid ceiling).  ``submission`` is the participant CSV with
columns ``n, square_id, x, y, angle``.

As a side effect it writes ``submission_details.csv``, one row per puzzle with
columns ``n`` and ``L``::

    n,L
    1,1.00000000000000000000000000000000000000000000000000
    2,2.00000000000000000000000000000000000000000000000000
    ...

``L`` is the measured bounding side for puzzle n -- not the per-puzzle score --
written with exactly 50 digits after the decimal point.  Those digits are
significant: L is measured in ``L_DPS``-digit mpmath arithmetic straight from
the submitted decimal strings, not from a double.  Puzzles with no accepted L
(absent from the submission, unparseable, failing column validation, or
rejected as infeasible) are written as zero.
"""

import contextlib
import csv
import io
import math
import multiprocessing
import re
from collections import defaultdict
from collections.abc import Callable, Sequence
from dataclasses import dataclass, field
from fractions import Fraction
from pathlib import Path

import mpmath
import pandas as pd

# ---------------------------------------------------------------------------
# interval
# ---------------------------------------------------------------------------

IVal = mpmath.ctx_iv.ivmpf

# Working precision for the (non-interval) measurement of the bounding side L,
# and the number of digits after the decimal point written for it in
# submission_details.csv.  The margin between the two keeps every written digit
# correct: the only rounding is in the trig of the corner computation, tens of
# digits below the last digit reported.
L_DPS = 60
L_DECIMALS = 50


@contextlib.contextmanager
def working_dps(dps: int):
    """Run a block with both mpmath contexts at ``dps`` digits, then restore them.

    ``mpmath.iv`` carries its own precision, independent of ``mpmath.mp``:
    setting only ``mpmath.mp.dps`` leaves every interval operation at the
    default 53 bits, which caps what a contact can be certified to at ~1e-13 no
    matter how far the escalation sequence climbs.  Both contexts are set here
    because the interval work is interleaved with plain ``mp`` arithmetic.

    >>> with working_dps(500):
    ...     width = mpmath.mpf(mpmath.iv.mpf('0.1').delta)
    >>> width < mpmath.mpf(10) ** -500
    True
    """
    old_iv_dps = mpmath.iv.dps
    old_mp_dps = mpmath.mp.dps
    try:
        mpmath.iv.dps = dps
        mpmath.mp.dps = dps
        yield
    finally:
        mpmath.iv.dps = old_iv_dps
        mpmath.mp.dps = old_mp_dps


def make_interval(value: str, dps: int = 50) -> IVal:
    with working_dps(dps):
        return mpmath.iv.mpf(value)


def iv_cos(x: IVal) -> IVal:
    return mpmath.iv.cos(x)


def iv_sin(x: IVal) -> IVal:
    return mpmath.iv.sin(x)


_ESCALATION_SEQUENCE = [50, 100, 200, 500]


def escalate(
    fn: Callable[[int], bool | None],
    max_dps: int = 500,
    start_dps: int = 50,
) -> bool | None:
    for dps in _ESCALATION_SEQUENCE:
        if dps < start_dps:
            continue
        if dps > max_dps:
            break
        result = fn(dps)
        if result is not None:
            return result
    return None


def iv_le(a: IVal, b: IVal) -> bool | None:
    """Certified less-or-equal: True if a <= b, False if a > b, None if undecided.

    Comparing two overlapping intervals is indeterminate.  mpmath 1.3 reports
    that as None, 1.4 raises ValueError; both mean "undecided" here.
    """
    try:
        return a <= b
    except ValueError:
        return None


def iv_min(values: Sequence[IVal]) -> IVal:
    """Interval extension of ``min``: the tightest interval containing the true minimum.

    Comparison operators are useless for this -- two overlapping intervals have
    no determinate order -- so take it endpoint-wise instead.  min of the lower
    bounds is a lower bound on the true min, and min of the upper bounds is an
    upper bound on it, whichever element turns out to be smallest.
    """
    return mpmath.iv.mpf([min(v.a for v in values), min(v.b for v in values)])


def iv_max(values: Sequence[IVal]) -> IVal:
    """Interval extension of ``max``. See :func:`iv_min`."""
    return mpmath.iv.mpf([max(v.a for v in values), max(v.b for v in values)])


# ---------------------------------------------------------------------------
# parser
# ---------------------------------------------------------------------------

# Bounds the cost of parsing one coordinate.  Far above any useful precision
# (L is reported to 50 decimals), so no honest submission comes near it.
_MAX_LITERAL_CHARS = 1000

_DECIMAL_RE = re.compile(r'^[+-]?(\d+\.?\d*|\.\d+)([eE][+-]?\d+)?$')


def _is_decimal(s: str) -> bool:
    return len(s) <= _MAX_LITERAL_CHARS and bool(_DECIMAL_RE.match(s.strip()))


def _check_decimal(s: str) -> str:
    """Return the stripped literal, or raise if it is not a plain decimal.

    >>> _check_decimal(" 2.70710678118654752440 ")
    '2.70710678118654752440'
    >>> _check_decimal("1.5e-3")
    '1.5e-3'
    >>> _check_decimal("1 + sqrt(2)")
    Traceback (most recent call last):
        ...
    ValueError: not a decimal number: '1 + sqrt(2)'
    """
    s = s.strip()
    if not _is_decimal(s):
        shown = s if len(s) <= 40 else s[:40] + '...'
        raise ValueError(f'not a decimal number: {shown!r}')
    return s


def parse_value(s: str, dps: int = 50) -> IVal:
    """Parse a decimal literal into an enclosing interval at ``dps`` digits."""
    return make_interval(_check_decimal(s), dps)


def parse_value_mp(s: str, dps: int = L_DPS) -> mpmath.mpf:
    """Parse a decimal literal into an ``mpf`` at ``dps`` digits.

    >>> mpmath.nstr(parse_value_mp("2.70710678118654752440"), 21, strip_zeros=False)
    '2.70710678118654752440'
    """
    s = _check_decimal(s)
    old_dps = mpmath.mp.dps
    try:
        mpmath.mp.dps = dps
        return mpmath.mpf(s)
    finally:
        mpmath.mp.dps = old_dps


def parse_value_float(s: str) -> float:
    """Parse a decimal literal to float.

    >>> parse_value_float("3.5")
    3.5
    >>> parse_value_float("-2.5")
    -2.5
    >>> parse_value_float("1.5e-3")
    0.0015

    Values too large for a double are rejected rather than becoming ``inf``,
    which would poison every downstream geometry check:

    >>> parse_value_float("1e400")
    Traceback (most recent call last):
        ...
    ValueError: value out of range for a double: '1e400'
    """
    s = _check_decimal(s)
    value = float(s)
    if not math.isfinite(value):
        raise ValueError(f'value out of range for a double: {s!r}')
    return value


# ---------------------------------------------------------------------------
# scoring
# ---------------------------------------------------------------------------


def score_puzzle(n: int, s_certified: float) -> float:
    """Score a single puzzle: L / sqrt(n) - 1 (lower is better).

    Result is >= 0 for any feasible packing (equality only for perfect-square
    grid packings).  The best-known side length s* plays no part: beating a
    record simply yields a smaller excess.

    Perfect-square grid packing scores 0 exactly:
    >>> score_puzzle(4, 2.0)
    0.0

    Larger L → larger (worse) score:
    >>> round(score_puzzle(5, 2.707), 6)
    0.210607

    Doubling L takes the score from 0 to 1:
    >>> score_puzzle(4, 4.0)
    1.0
    """
    L_B = math.sqrt(n)
    return s_certified / L_B - 1.0


def infeasible_score(n: int) -> float:
    """Penalty score for an infeasible / invalid submission for puzzle n.

    Equal to submitting the trivial grid packing: L_0 / sqrt(n) - 1.
    """
    return math.ceil(math.sqrt(n)) / math.sqrt(n) - 1.0


# ---------------------------------------------------------------------------
# geometry
# ---------------------------------------------------------------------------


@dataclass
class Square:
    x: float
    y: float
    angle: float
    x_str: str = ''
    y_str: str = ''
    angle_str: str = ''
    _corners_float: list[tuple[float, float]] = field(default_factory=list, repr=False)

    def corners_float(self) -> list[tuple[float, float]]:
        if not self._corners_float:
            self._corners_float = compute_corners_float(self.x, self.y, self.angle)
        return self._corners_float


def compute_corners_float(cx: float, cy: float, angle: float) -> list[tuple[float, float]]:
    """Return the four corners of a unit square centered at (cx, cy) rotated by angle.

    >>> corners = compute_corners_float(0.5, 0.5, 0.0)
    >>> [(round(x, 1), round(y, 1)) for x, y in corners]
    [(1.0, 1.0), (0.0, 1.0), (0.0, 0.0), (1.0, 0.0)]
    """
    c = math.cos(angle)
    s = math.sin(angle)
    h = 0.5
    dx1, dy1 = h * c - h * s, h * s + h * c
    dx2, dy2 = -h * c - h * s, -h * s + h * c
    return [
        (cx + dx1, cy + dy1),
        (cx + dx2, cy + dy2),
        (cx - dx1, cy - dy1),
        (cx - dx2, cy - dy2),
    ]


def compute_corners_mp(cx: mpmath.mpf, cy: mpmath.mpf, angle: mpmath.mpf) -> list[tuple[mpmath.mpf, mpmath.mpf]]:
    """``compute_corners_float`` at the ambient ``mpmath.mp`` precision."""
    c = mpmath.cos(angle)
    s = mpmath.sin(angle)
    h = mpmath.mpf(1) / 2
    dx1, dy1 = h * c - h * s, h * s + h * c
    dx2, dy2 = -h * c - h * s, -h * s + h * c
    return [
        (cx + dx1, cy + dy1),
        (cx + dx2, cy + dy2),
        (cx - dx1, cy - dy1),
        (cx - dx2, cy - dy2),
    ]


def measure_side_mp(squares: list['Square'], dps: int = L_DPS) -> mpmath.mpf:
    """Bounding side ``max(max_x - min_x, max_y - min_y)`` at ``dps`` digits.

    Measured from the submitted decimal strings, so a packing whose coordinates
    carry more than double precision keeps that precision in the reported L.

    >>> unit = Square(0.5, 0.5, 0.0, x_str="0.5", y_str="0.5", angle_str="0")
    >>> measure_side_mp([unit])
    mpf('1.0')
    """
    old_dps = mpmath.mp.dps
    try:
        mpmath.mp.dps = dps
        xs: list[mpmath.mpf] = []
        ys: list[mpmath.mpf] = []
        for sq in squares:
            corners = compute_corners_mp(
                parse_value_mp(sq.x_str, dps),
                parse_value_mp(sq.y_str, dps),
                parse_value_mp(sq.angle_str, dps),
            )
            for x, y in corners:
                xs.append(x)
                ys.append(y)
        return max(max(xs) - min(xs), max(ys) - min(ys))
    finally:
        mpmath.mp.dps = old_dps


def compute_corners_interval(cx_iv: IVal, cy_iv: IVal, angle_iv: IVal) -> list[tuple[IVal, IVal]]:
    c = iv_cos(angle_iv)
    s = iv_sin(angle_iv)
    h = mpmath.iv.mpf('0.5')
    dx1, dy1 = h * c - h * s, h * s + h * c
    dx2, dy2 = -h * c - h * s, -h * s + h * c
    return [
        (cx_iv + dx1, cy_iv + dy1),
        (cx_iv + dx2, cy_iv + dy2),
        (cx_iv - dx1, cy_iv - dy1),
        (cx_iv - dx2, cy_iv - dy2),
    ]


def compute_aabb(corners: list[tuple[float, float]]) -> tuple[float, float, float, float]:
    xs = [c[0] for c in corners]
    ys = [c[1] for c in corners]
    return (min(xs), min(ys), max(xs), max(ys))


def aabb_overlap(
    a: tuple[float, float, float, float],
    b: tuple[float, float, float, float],
) -> bool:
    return not (a[2] < b[0] or b[2] < a[0] or a[3] < b[1] or b[3] < a[1])


def _project_corners_float(corners: list[tuple[float, float]], ax: float, ay: float) -> tuple[float, float]:
    projs = [ax * cx + ay * cy for cx, cy in corners]
    return (min(projs), max(projs))


def sat_check_float(sq1: Square, sq2: Square, undecided_tol: float = 1e-8) -> str:
    """Check overlap between two unit squares using the Separating Axis Theorem.

    Returns ``"separated"``, ``"overlapping"``, or ``"undecided"``.

    >>> sat_check_float(Square(0.5, 0.5, 0.0), Square(5.0, 5.0, 0.0))
    'separated'
    >>> sat_check_float(Square(0.5, 0.5, 0.0), Square(0.6, 0.6, 0.0))
    'overlapping'
    """
    c1 = sq1.corners_float()
    c2 = sq2.corners_float()

    axes = _get_axes_float(sq1.angle, sq2.angle)

    min_overlap = float('inf')
    for ax, ay in axes:
        min1, max1 = _project_corners_float(c1, ax, ay)
        min2, max2 = _project_corners_float(c2, ax, ay)
        overlap = min(max1, max2) - max(min1, min2)
        if overlap < -undecided_tol:
            return 'separated'
        min_overlap = min(min_overlap, overlap)

    if min_overlap > undecided_tol:
        return 'overlapping'
    return 'undecided'


def _get_axes_float(angle1: float, angle2: float) -> list[tuple[float, float]]:
    c1, s1 = math.cos(angle1), math.sin(angle1)
    c2, s2 = math.cos(angle2), math.sin(angle2)
    return [(c1, s1), (-s1, c1), (c2, s2), (-s2, c2)]


def sat_check_interval(
    sq1_corners: list[tuple[IVal, IVal]],
    sq2_corners: list[tuple[IVal, IVal]],
    angle1_iv: IVal,
    angle2_iv: IVal,
) -> bool | None:
    c1_iv, s1_iv = iv_cos(angle1_iv), iv_sin(angle1_iv)
    c2_iv, s2_iv = iv_cos(angle2_iv), iv_sin(angle2_iv)
    axes = [(c1_iv, s1_iv), (-s1_iv, c1_iv), (c2_iv, s2_iv), (-s2_iv, c2_iv)]

    has_undecided = False
    for ax, ay in axes:
        projs1 = [ax * cx + ay * cy for cx, cy in sq1_corners]
        projs2 = [ax * cx + ay * cy for cx, cy in sq2_corners]

        # Endpoint-wise, via iv_min/iv_max: ordering two overlapping intervals
        # with < is indeterminate, and skipping the projections that compare
        # indeterminately would narrow the span -- which shrinks the overlap and
        # biases the test towards wrongly certifying a pair as separated.
        min1, max1 = iv_min(projs1), iv_max(projs1)
        min2, max2 = iv_min(projs2), iv_max(projs2)

        overlap = iv_min([max1, max2]) - iv_max([min1, min2])
        le_result = iv_le(overlap, mpmath.iv.mpf('0'))
        if le_result is True:
            return True
        if le_result is None:
            has_undecided = True

    if has_undecided:
        return None
    return False


def sort_and_sweep(squares: list[Square]) -> list[tuple[int, int]]:
    n = len(squares)
    if n <= 1:
        return []

    aabbs = []
    for i, sq in enumerate(squares):
        corners = sq.corners_float()
        aabb = compute_aabb(corners)
        aabbs.append((i, aabb))

    aabbs.sort(key=lambda x: x[1][0])

    pairs = []
    for i_idx in range(n):
        i, aabb_i = aabbs[i_idx]
        for j_idx in range(i_idx + 1, n):
            j, aabb_j = aabbs[j_idx]
            if aabb_j[0] > aabb_i[2]:
                break
            if aabb_overlap(aabb_i, aabb_j):
                pairs.append((i, j) if i < j else (j, i))

    return pairs


def verify_puzzle(
    squares: list[Square],
) -> tuple[bool, str]:
    """Check that no two squares overlap, using float SAT then interval arithmetic.

    >>> s1 = Square(0.5, 0.5, 0.0, x_str="0.5", y_str="0.5", angle_str="0")
    >>> s2 = Square(1.5, 0.5, 0.0, x_str="1.5", y_str="0.5", angle_str="0")
    >>> verify_puzzle([s1, s2])
    (True, 'feasible')
    >>> verify_puzzle([Square(0.5, 0.5, 0.0), Square(0.6, 0.6, 0.0)])
    (False, 'Squares 0 and 1 overlap (float check)')
    """
    pairs = sort_and_sweep(squares)
    undecided_pairs = []
    for i, j in pairs:
        result = sat_check_float(squares[i], squares[j])
        if result == 'overlapping':
            return (False, f'Squares {i} and {j} overlap (float check)')
        if result == 'undecided':
            undecided_pairs.append((i, j))

    if undecided_pairs:

        def check_overlap_certified(dps: int) -> bool | None:
            # The whole pair loop runs inside the precision context: parsing at dps
            # digits is not enough, since every later interval operation widens its
            # result to whatever precision mpmath.iv is set to at the time.
            with working_dps(dps):
                still_undecided = []
                for i, j in undecided_pairs:
                    sq_i, sq_j = squares[i], squares[j]
                    cx_i = parse_value(sq_i.x_str, dps)
                    cy_i = parse_value(sq_i.y_str, dps)
                    ai = parse_value(sq_i.angle_str, dps)
                    cx_j = parse_value(sq_j.x_str, dps)
                    cy_j = parse_value(sq_j.y_str, dps)
                    aj = parse_value(sq_j.angle_str, dps)
                    corners_i = compute_corners_interval(cx_i, cy_i, ai)
                    corners_j = compute_corners_interval(cx_j, cy_j, aj)
                    result = sat_check_interval(corners_i, corners_j, ai, aj)
                    if result is False:
                        return False
                    if result is None:
                        still_undecided.append((i, j))
                if still_undecided:
                    return None
                return True

        result = escalate(check_overlap_certified)
        if result is False:
            return (False, 'Squares overlap (certified)')
        if result is None:
            return (False, 'Overlap undecidable at max precision')

    return (True, 'feasible')


# ---------------------------------------------------------------------------
# main / Kaggle entry point
# ---------------------------------------------------------------------------


class ParticipantVisibleError(Exception):
    pass


MAX_N = 1000
KAGGLE_CORES = 4
MIN_PUZZLES_FOR_MP = 4


def _parse_submission_csv(text: str) -> dict[int, list[dict]]:
    reader = csv.DictReader(io.StringIO(text))
    required_cols = {'n', 'square_id', 'x', 'y', 'angle'}
    if not required_cols.issubset(set(reader.fieldnames or [])):
        missing = required_cols - set(reader.fieldnames or [])
        raise ParticipantVisibleError(f'Missing columns: {missing}')

    puzzles: dict[int, list[dict]] = defaultdict(list)
    for row_num, row in enumerate(reader, start=2):
        try:
            n = int(row['n'])
        except (ValueError, TypeError):
            raise ParticipantVisibleError(f'Row {row_num}: invalid n value: {row["n"]!r}')
        if n < 1 or n > MAX_N:
            raise ParticipantVisibleError(f'Row {row_num}: n={n} out of range [1, {MAX_N}]')
        puzzles[n].append(row)

    return dict(puzzles)


def _validate_puzzle(n: int, rows: list[dict]) -> str | None:
    if len(rows) != n:
        return f'n={n}: expected {n} rows, got {len(rows)}'

    square_ids = set()
    for row in rows:
        try:
            sid = int(row['square_id'])
        except (ValueError, TypeError):
            return f'n={n}: invalid square_id: {row["square_id"]!r}'
        square_ids.add(sid)

    expected_ids = set(range(n))
    if square_ids != expected_ids:
        return f'n={n}: square_ids should be 0..{n - 1}, got {sorted(square_ids)}'

    return None


def _build_squares(n: int, rows: list[dict]) -> tuple[list[Square], mpmath.mpf]:
    rows_by_id = {int(r['square_id']): r for r in rows}

    squares = []
    for sid in range(n):
        row = rows_by_id[sid]
        x_str = row['x'].strip()
        y_str = row['y'].strip()
        angle_str = row['angle'].strip()

        try:
            x_f = parse_value_float(x_str)
            y_f = parse_value_float(y_str)
            angle_f = parse_value_float(angle_str)
        except ValueError as e:
            raise ParticipantVisibleError(f'n={n}, square {sid}: cannot parse coordinates: {e}')

        squares.append(
            Square(
                x=x_f,
                y=y_f,
                angle=angle_f,
                x_str=x_str,
                y_str=y_str,
                angle_str=angle_str,
            )
        )

    return squares, measure_side_mp(squares)


def evaluate_single_puzzle(n: int, rows: list[dict]) -> tuple[int, float, str | None, str]:
    """Evaluate one puzzle, returning ``(n, score, L, reason)``.

    ``L`` is decimal text with ``L_DECIMALS`` digits after the point, and is
    reported only for a packing that passed the feasibility check: an
    unparseable, malformed, or overlapping puzzle gets None, so no L is
    published for a packing the metric rejected.  Text rather than a float
    because a double cannot hold those digits; the score itself is a double,
    taken from the same high-precision measurement.
    """
    error = _validate_puzzle(n, rows)
    if error:
        return (n, infeasible_score(n), None, error)

    try:
        squares, s_mp = _build_squares(n, rows)
    except ParticipantVisibleError as e:
        return (n, infeasible_score(n), None, str(e))

    feasible, reason = verify_puzzle(squares)
    if not feasible:
        return (n, infeasible_score(n), None, f'infeasible: {reason}')

    puzzle_score = score_puzzle(n, float(s_mp))
    return (n, puzzle_score, _format_detail(s_mp), 'ok')


def _evaluate_puzzle_worker(args: tuple) -> tuple[int, float, str | None, str]:
    n, rows = args
    try:
        return evaluate_single_puzzle(n, rows)
    except Exception as e:  # noqa: BLE001 - one bad puzzle must not kill the pool
        return (n, infeasible_score(n), None, f'worker error: {e}')


def _evaluate_puzzles_parallel(
    puzzles: dict[int, list[dict]],
    puzzle_ns: set[int],
) -> tuple[float, list[tuple[int, float, str | None, str]]]:
    """Score every n in ``puzzle_ns``; rows for any other n are ignored."""
    work = [(n, rows) for n, rows in sorted(puzzles.items()) if n in puzzle_ns]

    if not work:
        results: list[tuple[int, float, str | None, str]] = []
    elif len(work) < MIN_PUZZLES_FOR_MP:
        results = [evaluate_single_puzzle(n, rows) for n, rows in work]
    else:
        with multiprocessing.Pool(processes=KAGGLE_CORES) as pool:
            results = pool.map(_evaluate_puzzle_worker, work)

    # Any scored n missing from the submission gets the grid-ceiling penalty
    # (lower-is-better metric — otherwise omitting a puzzle would improve the
    # total).
    seen = {n for n, _, _, _ in results}
    for n in puzzle_ns:
        if n not in seen:
            results.append((n, infeasible_score(n), None, 'missing from submission'))
    results.sort(key=lambda r: r[0])

    total = sum(puzzle_score for _, puzzle_score, _, _ in results)
    return total, results


def _format_detail(value: mpmath.mpf) -> str:
    """Decimal text for one per-puzzle L: exactly ``L_DECIMALS`` digits after the point.

    An ``mpf`` is a binary fraction (``man * 2**exp``), so it is converted
    exactly to a ``Fraction`` and rounded once, half-to-even, at the last
    written digit.  ``mpmath.nstr`` counts significant digits rather than
    decimal places, and ``format(mpf, '.50f')`` needs mpmath >= 1.4, so neither
    is usable here.

    >>> _format_detail(mpmath.mpf(2))[:12]
    '2.0000000000'
    >>> len(_format_detail(mpmath.mpf(2)).split('.')[1])
    50
    >>> _format_detail(parse_value_mp('1.41421356237309504880168872420969807856967187537695'))
    '1.41421356237309504880168872420969807856967187537695'
    """
    exact = Fraction(value.man) * Fraction(2) ** value.exp
    scale = 10**L_DECIMALS
    units = round(exact * scale)
    sign = '-' if units < 0 else ''
    whole, frac = divmod(abs(units), scale)
    return f'{sign}{whole}.{frac:0{L_DECIMALS}d}'


def score(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str,
) -> float:
    """Kaggle entry point. Score a submission against the solution's puzzle set.

    ``solution`` is read for one thing only: the set of n to score.  Everything
    else about a puzzle -- best-known side length, proven-optimal flag, source
    -- is irrelevant to ``L / sqrt(n) - 1``.

    >>> import pandas as pd
    >>> sol = pd.DataFrame({"id": [0], "n": [1]})
    >>> sub = pd.DataFrame({"id": [0], "n": [1], "square_id": [0],
    ...     "x": ["0.5"], "y": ["0.5"], "angle": ["0"]})
    >>> score(sol, sub, "id")  # single unit square, L=1, L/sqrt(1) - 1 = 0.0
    0.0
    """
    solution = solution.drop(columns=[row_id_column_name])
    submission = submission.drop(columns=[row_id_column_name])

    puzzle_ns = {int(n) for n in solution['n']}

    text = submission.to_csv(index=False)
    puzzles = _parse_submission_csv(text)
    total, results = _evaluate_puzzles_parallel(puzzles, puzzle_ns)

    # L is the measured bounding side for puzzle n -- NOT the per-puzzle score
    # L/sqrt(n) - 1.  Puzzles with no accepted L (missing from the submission,
    # unparseable, failing column validation, or rejected as infeasible) keep
    # the historical 0 placeholder: an L measured off an overlapping packing is
    # not a real packing of n squares, and downstream harvesting treats every
    # published L as attainable.  Values arrive here already rendered as decimal
    # text with L_DECIMALS digits after the point; unlike a JSON float they are
    # not re-serialized downstream, so all 50 digits reach the stored file
    # verbatim.
    result_map = {n: L_text for n, _, L_text, _ in results if L_text is not None}
    zero = _format_detail(mpmath.mpf(0))

    output_dir = Path('/kaggle/working')
    if not output_dir.is_dir():
        output_dir = Path('.')
    with open(output_dir / 'submission_details.csv', 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['n', 'L'])
        for n in range(1, MAX_N + 1):
            writer.writerow([n, result_map.get(n, zero)])

    return float(total)


### research/validate_candidate.py

In [ ]:
%%writefile research/validate_candidate.py
import csv, hashlib, json, math, sys, time
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor
sys.path.insert(0,str(Path(__file__).resolve().parent))
import official_metric as metric

if __name__=='__main__':
    start=time.perf_counter();path=Path(sys.argv[1] if len(sys.argv)>1 else 'artifacts/submission.csv')
    puzzles=defaultdict(list)
    with path.open(newline='') as f:
        rows=list(csv.DictReader(f))
    assert [int(r['id']) for r in rows]==list(range(len(rows)))
    for r in rows:puzzles[int(r['n'])].append(r)
    with ProcessPoolExecutor(max_workers=8) as ex:
        results=list(ex.map(metric._evaluate_puzzle_worker,sorted(puzzles.items())))
    rejected=[r for r in results if r[3]!='ok']
    total=sum(r[1] for r in results)+sum(metric.infeasible_score(n) for n in range(1,1001) if n not in puzzles)
    record={'sha256':hashlib.sha256(path.read_bytes()).hexdigest(),'file':str(path),'metric_version':17,'rows':len(rows),'puzzles':len(puzzles),'rejected':rejected,'score':total,'grid_score':sum(metric.infeasible_score(n) for n in range(1,1001)),'elapsed_seconds':time.perf_counter()-start}
    report=Path(sys.argv[2] if len(sys.argv)>2 else 'artifacts/validation.json')
    report.write_text(json.dumps(record,indent=2))
    report.with_name(report.stem+'_puzzle_scores.json').write_text(json.dumps(results))
    print(json.dumps({**record,'rejected':rejected[:3],'rejected_count':len(rejected)},indent=2),flush=True)
    assert not rejected,f'{len(rejected)} rejected puzzles'


### research/top1-refresh/kirill_sources/packing_geometry.py

In [ ]:
%%writefile research/top1-refresh/kirill_sources/packing_geometry.py
"""CPU-only geometry and file utilities for the Squares in Squares experiment."""
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
import math
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree, ConvexHull
from scipy.optimize import minimize_scalar


def grid(n):
    k = math.ceil(math.sqrt(n))
    return np.array([(i % k + .5, i // k + .5, 0.) for i in range(n)])


def aabbs(p):
    h = (np.abs(np.cos(p[:, 2])) + np.abs(np.sin(p[:, 2]))) / 2
    return p[:, :2] - h[:, None], p[:, :2] + h[:, None]


def side(p):
    lo, hi = aabbs(p)
    return float(np.max(hi.max(0) - lo.min(0)))


def normalize(p):
    p = np.array(p, dtype=float, copy=True)
    p[:, :2] -= aabbs(p)[0].min(0)
    return p


def rotate(p, angle):
    p = p.copy()
    c, s = math.cos(angle), math.sin(angle)
    p[:, :2] = p[:, :2] @ np.array([[c, s], [-s, c]])
    p[:, 2] += angle
    return normalize(p)


def vertices(p):
    u = np.array([[-.5, -.5], [.5, -.5], [.5, .5], [-.5, .5]])
    c, s = np.cos(p[:, 2]), np.sin(p[:, 2])
    return p[:, None, :2] + np.stack((c[:, None]*u[:, 0]-s[:, None]*u[:, 1],
                                     s[:, None]*u[:, 0]+c[:, None]*u[:, 1]), 2)


def pair_data(p, radius=1.55):
    pairs = cKDTree(p[:, :2]).query_pairs(radius, output_type="ndarray")
    if not len(pairs):
        return pairs, np.empty((0, 4)), np.empty(0)
    i, j = pairs.T
    d = p[j, :2] - p[i, :2]
    a, b = p[i, 2], p[j, 2]
    projections = np.column_stack((d[:, 0]*np.cos(a)+d[:, 1]*np.sin(a),
                                    -d[:, 0]*np.sin(a)+d[:, 1]*np.cos(a),
                                    d[:, 0]*np.cos(b)+d[:, 1]*np.sin(b),
                                    -d[:, 0]*np.sin(b)+d[:, 1]*np.cos(b)))
    support = (1 + np.abs(np.cos(a-b)) + np.abs(np.sin(a-b))) / 2
    return pairs, projections, support


def min_gap(p):
    _, projection, support = pair_data(p)
    return float(np.min(np.abs(projection).max(1)-support)) if len(support) else math.inf


def repair(p, clearance=3e-10):
    """Separate centres slightly without ever changing unit-square dimensions."""
    p = normalize(p)
    _, projection, support = pair_data(p)
    if len(support):
        ratio = float(np.max((support+clearance)/np.maximum(np.abs(projection).max(1), 1e-30)))
        if ratio > 1.005:
            return None
        p[:, :2] *= max(1., ratio)
    return normalize(p)


def load_bank(path="inputs/submission.csv"):
    data = pd.read_csv(path, float_precision="round_trip")
    bank = {n: grid(n) for n in range(1, 1001)}
    for n, group in data.groupby("n"):
        n = int(n)
        assert len(group) == n
        bank[n] = group.sort_values("square_id")[["x", "y", "angle"]].to_numpy(float)
    return bank


def save_bank(path, bank):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, **{str(n): p for n, p in bank.items()})


def total_score(bank):
    return float(sum(side(bank[n])/math.sqrt(n)-1 for n in range(1, 1001)))


def optimal_rotation(p):
    """Minimize the enclosing square of the fixed full layout over global angle."""
    points = vertices(p).reshape(-1, 2)
    hull = points[ConvexHull(points).vertices]
    angles = np.linspace(-math.pi/4, math.pi/4, 721)
    c, s = np.cos(angles), np.sin(angles)
    xx = hull[:, 0, None]*c - hull[:, 1, None]*s
    yy = hull[:, 0, None]*s + hull[:, 1, None]*c
    lengths = np.maximum(np.ptp(xx, axis=0), np.ptp(yy, axis=0))
    indices = np.flatnonzero((lengths[1:-1] <= lengths[:-2]) &
                             (lengths[1:-1] <= lengths[2:])) + 1
    winner = (side(p), 0.)
    def objective(a):
        c, s = math.cos(a), math.sin(a)
        return max(np.ptp(hull[:, 0]*c-hull[:, 1]*s),
                   np.ptp(hull[:, 0]*s+hull[:, 1]*c))
    for k in indices:
        fit = minimize_scalar(objective, bounds=(angles[k-1], angles[k+1]),
                              method="bounded", options={"xatol": 1e-14})
        if fit.fun < winner[0]:
            winner = (float(fit.fun), float(fit.x))
    return rotate(p, winner[1]), winner[1]


### research/top1-refresh/admit.py

In [ ]:
%%writefile research/top1-refresh/admit.py
"""CPU-only monotone admission with unchanged official metric and pandas replay."""
import os
for k in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','VECLIB_MAXIMUM_THREADS']:os.environ[k]='1'
import sys,csv,json,hashlib,io,argparse,math
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
import numpy as np
R=Path(__file__).resolve().parents[2];sys.path.insert(0,str(R/'research'));sys.path.insert(0,str(Path(__file__).parent/'kirill_sources'))
import official_metric as metric
from packing_geometry import side,repair
def groups(p):
 d={}
 with Path(p).open(newline='')as f:
  for r in csv.DictReader(f):d.setdefault(int(r['n']),[]).append(r)
 return d
def array(rows):return np.array([[float(r[k])for k in ['x','y','angle']]for r in rows])
def rows(n,p):return [dict(n=str(n),square_id=str(i),x=format(x,'.17g'),y=format(y,'.17g'),angle=format(a,'.17g'))for i,(x,y,a)in enumerate(p)]
def check(job):
 n,new,old=job;a=metric._evaluate_puzzle_worker((n,old));b=metric._evaluate_puzzle_worker((n,new))
 rec=dict(n=n,before=float(a[2]),after=float(b[2])if b[2]is not None else None,gain=a[1]-b[1],status=b[3])
 if b[3]!='ok'or rec['gain']<1e-10:return n,None,rec
 import pandas as pd
 text=io.StringIO();pd.read_csv(io.StringIO(pd.DataFrame(new).to_csv(index=False))).to_csv(text,index=False,float_format='%.17g');text.seek(0)
 rt=metric._evaluate_puzzle_worker((n,list(csv.DictReader(text))))
 if rt[3]!='ok'or abs(rt[1]-b[1])>1e-10:return n,None,dict(**rec,roundtrip_rejected=rt[3])
 return n,new,rec
def write(path,bank):
 with Path(path).open('w',newline='')as f:
  w=csv.writer(f);w.writerow(['id','n','square_id','x','y','angle']);i=0
  for n,rr in sorted(bank.items()):
   for r in rr:w.writerow([i,n,r['square_id'],r['x'],r['y'],r['angle']]);i+=1
if __name__=='__main__':
 a=argparse.ArgumentParser();a.add_argument('--baseline',required=True);a.add_argument('--sources',nargs='+',required=True);a.add_argument('--tag',required=True);a.add_argument('--include-new',action='store_true');x=a.parse_args();bank=groups(x.baseline);records=[];rejected=[];stages=[]
 for source in x.sources:
  p=Path(source)
  if p.suffix=='.npz':
   z=np.load(p);candidate={}
   for key in z.files:
    n=int(key);q=repair(z[key],clearance=4e-10)
    if q is not None:candidate[n]=rows(n,q)
  else:candidate=groups(p)
  tasks=[]
  for n,rr in candidate.items():
   old=bank.get(n)
   if old is None and x.include_new and 1<=n<=1000:
    k=math.ceil(math.sqrt(n))
    old=rows(n,np.array([(i%k+.5,i//k+.5,0.)for i in range(n)]))
   if old is not None and side(array(rr))<side(array(old))-1e-8:tasks.append((n,rr,old))
  kept=[]
  with ProcessPoolExecutor(max_workers=2)as pool:
   for n,new,r in pool.map(check,tasks):
    r['stage']=p.stem
    if new is not None:bank[n]=new;kept.append(r);records.append(r)
    else:rejected.append(r)
  stage=dict(stage=p.stem,accepted=len(kept),gain=sum(r['gain']for r in kept),sha256=hashlib.sha256(p.read_bytes()).hexdigest());stages.append(stage);print(json.dumps(stage),flush=True)
 out=R/'artifacts'/f'{x.tag}_submission.csv';write(out,bank);report=dict(baseline=x.baseline,sha256=hashlib.sha256(out.read_bytes()).hexdigest(),gain=sum(r['gain']for r in records),stages=stages,records=records,rejected=rejected)
 (R/'research'/f'{x.tag}.json').write_text(json.dumps(report,indent=2));print(json.dumps({k:v for k,v in report.items()if k not in ['records','rejected']}),flush=True)


In [ ]:
source_formats={'solver.py': {'sha256': '923ce87cf1c4b2a7fc43e10220b1ef2ce4be02afdf52733a498641061b12f250', 'crlf': False, 'trailing': True}, 'packing_pipeline.py': {'sha256': 'b484c774f27724f4c75da950047e61347c0d14df0279ad04cb46d770e628f4f4', 'crlf': False, 'trailing': True}, 'hybrid_search.py': {'sha256': '1215d276fa6b638c002bd6c1005646f29c032c8ae3e1fe4cc657499708b0339c', 'crlf': False, 'trailing': True}, 'gpu_families.py': {'sha256': 'df4f37b43ca3aae13cd0329f5d20356811f9adf7c685af7d706f00e0502d8545', 'crlf': False, 'trailing': True}, 'gpu_topology.py': {'sha256': '93e352b406e100d6ae2a5eb2f8dc4dfa77df3a84da7cd92d6bb0fa990a700eb7', 'crlf': False, 'trailing': True}, 'gpu_rect_optimizer.py': {'sha256': '1e9214275e5ee750ad3b880813d42679f4b22674a5f927af568bce99e5567a7b', 'crlf': False, 'trailing': True}, 'gpu_rect_explore.py': {'sha256': '2ca2a9586c9d758a66527c44fe68e34cd8f6e33f1216979f959b40597b0c71c4', 'crlf': False, 'trailing': True}, 'gpu_upright_rows.py': {'sha256': '7560511e30411dcd09c508b0783b66a19a493a39f2a40954181c089ecd46330c', 'crlf': False, 'trailing': True}, 'upright_refill.py': {'sha256': '2ca7759245c7eb65142319fd70c3c1323e2cbe6c9793f16d131fa6f01e7f79fd', 'crlf': False, 'trailing': True}, 'gpu_ftmc_refill.py': {'sha256': 'ea6a1487285c43e822066a95abe336d09556eed9e80296d6a2366ee03450560c', 'crlf': False, 'trailing': True}, 'gpu_fault_composition.py': {'sha256': 'd658274c60beaf0df8129054328ad676adc1215b32a843b21ecf27a17ab89785', 'crlf': False, 'trailing': True}, 'gpu_seam_profiles.py': {'sha256': '0319053b7608a02a9bec21a2cb5ba022f05cd04f3a7c7ce4d2e4cdce0051ec2c', 'crlf': False, 'trailing': True}, 'gpu_adaptive_seams.py': {'sha256': 'b2b83f5c725655df188a1b70d8b13a0123b7be8240836339d1ce38888b9f86c0', 'crlf': False, 'trailing': True}, 'gpu_projected_union.py': {'sha256': '15f027818d5d5258dd0121741db0ce6b0be73c2396249cc55c22c2770fc23a78', 'crlf': False, 'trailing': True}, 'research/official_metric.py': {'sha256': '312458227c78f56bd5e2eec3dc5fbb8787989422bbf72ce66c3c290048e89733', 'crlf': False, 'trailing': True}, 'research/validate_candidate.py': {'sha256': '9c473292c5b08993c5c67aa7cfdbe962689ca3c9407d5341754269b2cb9faf7f', 'crlf': False, 'trailing': True}, 'research/top1-refresh/kirill_sources/packing_geometry.py': {'sha256': '30c649ca0b03d0ef83d0c2530c098c9e24ef53fd1d0b1ac1d898f76d47d1f50e', 'crlf': True, 'trailing': True}, 'research/top1-refresh/admit.py': {'sha256': '6b9aea97bddd3ea3f9673f9bb0ba619e59c33eb3fbcf57fec932aca92bf6b571', 'crlf': False, 'trailing': True}}
for name,fmt in source_formats.items():
    p=Path(name);blob=p.read_bytes().replace(b'\r\n',b'\n')
    if fmt['crlf']:blob=blob.replace(b'\n',b'\r\n')
    if not fmt['trailing']:blob=blob.rstrip(b'\r\n')
    p.write_bytes(blob)
    assert hashlib.sha256(blob).hexdigest()==fmt['sha256'],name
print('All readable source cells restored byte-exactly.')

## 5. Optional bounded CUDA search and independent certification

Search is disabled below. The search stage is bounded to 400 seconds, followed by a bounded 160-second composition pass; startup and certification add overhead. Fresh stochastic trials can differ from the frozen campaign. The default certification re-evaluates every supplied puzzle with the unchanged official metric v17 and verifies an ordinary pandas float64 serialization. Missing sizes retain official grid fallback terms.

In [ ]:
import mpmath
if not hasattr(mpmath.mpf(1),'man'):subprocess.run([sys.executable,'-m','pip','install','-q','mpmath==1.4.1'],check=True)
assert hashlib.sha256(Path('research/official_metric.py').read_bytes()).hexdigest()=='312458227c78f56bd5e2eec3dc5fbb8787989422bbf72ce66c3c290048e89733'
if RUN_SEARCH:
    shutil.copyfile(DATA/'projected_public.csv','artifacts/update25_public_submission.csv')
    for name in ['fault_source_donors.npz','fault_source_labels.json']:shutil.copyfile(DATA/name,Path('artifacts')/name)
    shutil.copyfile(DATA/'separator_donor_gadgets.npz','artifacts/tip_frontier1_gadgets.npz');shutil.copyfile(DATA/'separator_crystal.npz','artifacts/crystal_polished1_gadgets.npz')
    subprocess.run([sys.executable,'-u','gpu_projected_union.py','--baseline','artifacts/update25_public_submission.csv','--tag','update25_projected','--seconds','400','--limit','80'],check=True,timeout=430)
    subprocess.run([sys.executable,'research/top1-refresh/admit.py','--baseline','submission.csv','--sources','artifacts/update25_projected_packings.npz','--tag','optional_projected'],check=True)
    subprocess.run([sys.executable,'-u','gpu_adaptive_seams.py','--baseline','artifacts/optional_projected_submission.csv','--seconds','160','--bins','24','--tag','optional_composed'],check=True,timeout=190)
    subprocess.run([sys.executable,'research/top1-refresh/admit.py','--baseline','artifacts/optional_projected_submission.csv','--sources','artifacts/optional_composed_packings.npz','--tag','optional_final'],check=True)
    shutil.copyfile('artifacts/optional_final_submission.csv','submission.csv')
for src,dest in [('submission.csv','validation.json'),('roundtrip.csv','roundtrip_validation.json')]:
    if src=='roundtrip.csv':pd.read_csv('submission.csv').to_csv(src,index=False,float_format='%.17g')
    result=subprocess.run([sys.executable,'-u','research/validate_candidate.py',src,dest],capture_output=True,text=True,check=True,timeout=300);print(result.stdout)
raw=json.loads(Path('validation.json').read_text());rt=json.loads(Path('roundtrip_validation.json').read_text());assert not raw['rejected']and not rt['rejected']and abs(raw['score']-rt['score'])<1e-10
assert raw['score']<=summary['score']+1e-10
if not RUN_SEARCH:assert raw['sha256']==summary['sha256']
print(f"Certified score: {raw['score']:.12f}");display(FileLink('submission.csv'))

## Credits and limits

Public witnesses and ideas: [Kirill Tymchishin](https://www.kaggle.com/code/kirilltym/squares-in-squares-cpu-cuda-mps-eda-projunion), [prvsiyan](https://www.kaggle.com/code/prvsiyan/auto-updater), avazaku, Chris Deotte, Alex Chilton, Aqib Raza, Malyshev Danil, Sung Heo, Dariush Afshar, Francisco Couzo, Joshua Levy, David Ellsworth, Erich Friedman and the complete inherited notice chain. Earlier smooth-line experiments follow Evan Daniel's fq formulation. Their work retains attribution and original licenses.

`projected_SOURCE_NOTICES.md`, `projected_kirill_NOTICES.md`, `projected_prvsiyan_credits.csv` and the earlier full notices accompany the frozen inputs. Public witness adoption is not claimed as independent discovery. LP branch selection and local CUDA search remain heuristic. Recorded LP wins are sequential within the pilot, not an exclusive causal comparison against a matched optimizer. Campaign wall time includes CPU restoration; kernel time is reported separately.